# 歩行周期を筋駆動の OpenSimAD で解く

OpenCapで撮った歩行から**歩行周期を1つ**切り出し，**筋駆動**の OpenSimAD で解きます．
床反力，関節モーメント，筋活動を求め，OpenSim形式で書き出します．

1 節の入力を設定し，「Run」→「Run All Cells」で上から順に実行します．
計算時間は区間の長さとメッシュ数によって変わります．

| 用語 | 意味 |
|---|---|
| 歩行周期 | 片足の踵接地から，同じ足の次の踵接地まで（1 秒ほど）．踵接地は Zeni et al.(2008) の座標の方法で求めます（踵のマーカーと骨盤の後ろの前後の差が最大になる時刻．原著は仙骨のマーカー，ここでは左右の上後腸骨棘の中点を使います） |
| OpenSimAD | 筋骨格モデルが入力の動き（OpenCap の関節角度）をなるべくなぞり，しかも物理的に成り立つ（運動方程式を満たす）ように，関節を動かす力を最適化で求める方法．床反力も，足の裏の接触のモデルから求めます．OpenSim のモデルで自動微分を使い，CasADi と組み合わせて軌道最適化を速く解く道具です(Falisse et al., 2019)|
| 筋駆動 | 脚の関節を筋（モデルの左右 80 の筋）の力で動かすモデル．筋活動（0〜1）が出ます．腰と腕は関節のトルクで動かします |
| メッシュ | 最適化で時間を区切る細かさ（1 秒あたりの区間の数）．50 なら，1.1 秒の歩行周期を 55 ほどの区間に区切ります |

**流れ**
1. 入力
2. 準備
3. データを読む
4. 区間を選ぶ
5. 解く
6. 結果を見る
7. OpenSim で読める形に書き出す
8. 保存



## 1. 入力

`/data` にある OpenCap の zip または展開済みフォルダを使います．
次のファイルが必要です（動画は不要です）．

| ファイル | 中身 |
|---|---|
| `MarkerData/<試行>.trc` | マーカーの位置（歩行のイベントと床の傾きに使う） |
| `OpenSimData/Kinematics/<試行>.mot` | 関節角度（OpenSimAD が追う入力） |
| `OpenSimData/Model/LaiUhlrich2022_scaled.osim` | 体に合わせたモデル（OpenCap の1台版では `OpenSimData/Model/<試行>/` の中） |
| `sessionMetadata.yaml` | 体重（`mass_kg`）と身長（`height_m`）．1台版にはないので，下の `MASS_KG`・`HEIGHT_M` に書く |

入力は `/data`，保存先は `/results` です．例：`SESSION = '/data/OpenCapData_abc.zip'`．

区間の選び方：
- `CYCLE = 'auto'`（既定）：TUG（立ち上がりと方向転換が見つかる試行）なら，往路の直進の区間で，立ち上がりの終わりのあとに始まり，
  方向転換の始まり（骨盤の向きの変化の 5 %）までに終わる歩行周期（両足にあれば終わりが早いほう）．
  ほかの歩行なら，いちばん長い直進の区間で，最初と最後の周期（歩き始め・歩き終わり）を除き，区間の真ん中にいちばん近い歩行周期．
- `CYCLE = <数>`：4 節の表の番号の歩行周期．
- `WINDOW = (開始の秒, 終了の秒)`：その区間（歩行周期でなくてもよい．`CYCLE` より優先）．0.3 秒より短い区間は解けません．

扱わないもの：トレッドミルの歩行（ベルトの速さを入れる仕組みがありません）．走行や跳躍（設定は歩行用です）．

In [ ]:
# ===== ここを書き換える（パスはコンテナの中のもの．データのフォルダは /data，結果のフォルダは /results） =====
SESSION = '/data/OpenCapData_xxxxxxxx.zip'        # /data 内の zip または展開済みフォルダ
TRIAL = ''                                        # 解く試行の名前（例 'walk01'）．空なら 3 節で試行の一覧を出して止まる
CYCLE = 'auto'                                    # 'auto'：自動で選ぶ．数なら 4 節の表の番号の歩行周期
WINDOW = None                                     # (開始の秒, 終了の秒) と書くと，その区間を解く（CYCLE より優先）
MASS_KG = None                                    # 体重（kg）．None なら sessionMetadata.yaml の値
HEIGHT_M = None                                   # 身長（m）．None なら sessionMetadata.yaml の値
MESH = 50                                         # 1 秒あたりの区間の数（25 にすると速いが粗い）
LEVEL_FLOOR = True                                # 解く前に，傾いて再構成された床を水平にする
TIME_LIMIT_MIN = 300                              # 求解の打ち切り（分）
OUT_DIR = '/results'                              # 結果を書くフォルダ

## 2. 準備

関数を読み込み，OpenSim の環境（Docker のイメージの中の `/opt/osad`：OpenSim 4.5，CasADi 3.7.2，opencap-processing の版 72b5416，
OpenSimAD のライブラリ）を確かめます．

In [ ]:
# 2. 準備：読み込み
import os, glob, json, shutil, time, zipfile, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option('display.precision', 3)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
TOOL_TEXT = {}

In [ ]:
# 関数：ファイルを読む，床，歩行のイベント，TUG の区切り
import warnings

from scipy import signal


def read_mot(path):
    """関節角度ファイル（.mot）を読む．列名は OpenCap の出力のまま（例：knee_angle_r，単位は度）．
    pelvis_tx，pelvis_ty，pelvis_tz（骨盤の位置）だけは m．"""
    with open(path, encoding='utf-8', errors='replace') as f:
        lines = f.read().splitlines()
    i = next(k for k, line in enumerate(lines) if line.strip().lower() == 'endheader')
    df = pd.read_csv(path, sep=r'\s+', skiprows=i + 1, engine='python')
    return df


# OpenCap Monocular（スマートフォン1台）の .trc はマーカー名が違う（_study が付かない）．2台以上のときの名前に読み替える
MONO_NAMES = {'r_ASIS': 'r.ASIS_study', 'l_ASIS': 'L.ASIS_study', 'r_PSIS': 'r.PSIS_study', 'l_PSIS': 'L.PSIS_study',
              'C7': 'C7_study', 'r_shoulder': 'r_shoulder_study', 'l_shoulder': 'L_shoulder_study',
              'r_elbow': 'r_lelbow_study', 'l_elbow': 'L_lelbow_study', 'r_melbow': 'r_melbow_study', 'l_melbow': 'L_melbow_study',
              'r_wrist_radius': 'r_lwrist_study', 'l_wrist_radius': 'L_lwrist_study',
              'r_wrist_ulna': 'r_mwrist_study', 'l_wrist_ulna': 'L_mwrist_study'}
MONO_NAMES.update({p + s: q + s + '_study' for s in ('knee', 'mknee', 'ankle', 'mankle', 'toe', '5meta', 'calc')
                   for p, q in (('r_', 'r_'), ('l_', 'L_'))})


def read_trc(path):
    """マーカー座標ファイル（.trc）を読む．
    戻り値は辞書：time（s），rate（Hz），markers（マーカー名 → N×3 の配列 [X, Y, Z]，m，Y が鉛直上向き），
    monocular（1台で処理したデータなら True）．
    名前が _study で終わるのが解剖学的マーカー（例：r_calc_study は右踵，r.PSIS_study は右上後腸骨棘）．
    1台で処理したデータ（OpenCap Monocular）は名前が違うので，同じ名前に読み替える（例：l_calc → L_calc_study）．"""
    with open(path, encoding='utf-8', errors='replace') as f:
        lines = f.read().splitlines()
    keys, vals = lines[1].split('\t'), lines[2].split('\t')
    rate = float(vals[keys.index('DataRate')])
    scale = 0.001 if vals[keys.index('Units')].strip() == 'mm' else 1.0
    names = [n.strip() for n in lines[3].split('\t')[2:] if n.strip()]
    rows = [line.split('\t') for line in lines[5:] if line.strip()]
    X = np.array([[float(v) if v.strip() else np.nan for v in r[:2 + 3 * len(names)]] for r in rows])
    mono = 'r_ASIS' in names and 'r.ASIS_study' not in names
    markers = {(MONO_NAMES.get(n, n) if mono else n): X[:, 2 + 3 * j:5 + 3 * j] * scale for j, n in enumerate(names)}
    return {'time': X[:, 1], 'rate': rate, 'markers': markers, 'monocular': mono}


def lowpass(x, fs, fc=6.0, order=4):
    """ゼロ位相のバターワース低域通過フィルタ（列ごと）．OpenCap の立ち上がりの解析は 6 Hz を使う．"""
    b, a = signal.butter(order, fc / (fs / 2))
    return signal.filtfilt(b, a, np.asarray(x, dtype=float), axis=0)


UP = np.array([0.0, 1.0, 0.0])


def pelvis_center(trc):
    """骨盤の中心：左右の上前・上後腸骨棘（4点）の平均"""
    m = trc['markers']
    return (m['r.ASIS_study'] + m['L.ASIS_study'] + m['r.PSIS_study'] + m['L.PSIS_study']) / 4


def body_axes(trc):
    """水平面内の前方向 fwd と右方向 right（単位ベクトル）．
    骨盤の向き（上後腸骨棘の中点 → 上前腸骨棘の中点）を，試行の中央の 50 % で平均して決める
    （OpenCap 公式の歩行解析と同じ）．試行中に向きを変える動作（方向転換など）には使えない．"""
    m = trc['markers']
    head = (m['r.ASIS_study'] + m['L.ASIS_study']) / 2 - (m['r.PSIS_study'] + m['L.PSIS_study']) / 2
    n = len(head)
    d = np.nanmean(head[n // 4:max(n // 4 + 1, 3 * n // 4)], axis=0)
    d[1] = 0.0
    fwd = d / np.linalg.norm(d)
    right = np.cross(fwd, UP)                        # OpenSim の座標では Z が右
    return fwd, right


def trunk_lean(trc):
    """体幹の傾き（度）：骨盤の中心 → 左右の肩峰の中点の線が鉛直となす角．
    sagittal は前傾が正，lateral は右へ傾くと正（水平面内の前・右方向は body_axes で決める）．"""
    fwd, right = body_axes(trc)
    m = trc['markers']
    v = (m['r_shoulder_study'] + m['L_shoulder_study']) / 2 - pelvis_center(trc)
    up = v @ UP
    return pd.DataFrame({'time': trc['time'],
                         'sagittal': np.degrees(np.arctan2(v @ fwd, up)),
                         'lateral': np.degrees(np.arctan2(v @ right, up))})

def floor_plane(trc, vthr=0.15, min_spread=1.0):
    """床面 y = a x + b z + c を，接地中（速さ vthr m/s 未満）の左右の踵マーカーから最小二乗で求める．
    キャリブレーションのチェッカーボードが傾いていたり小さく写っていたりすると，床が傾いて再構成される．
    踵が min_spread m 以上動いた向きの傾きだけを求める（その場の動作では傾きを求められず nan）．
    戻り値：a，b，c（c は踵マーカーの高さなので床より数 cm 上），傾き tilt_deg，平面からのずれ rms_m"""
    fs, P = trc['rate'], []
    for k in ('r_calc_study', 'L_calc_study'):
        x = lowpass(trc['markers'][k], fs)
        v = np.linalg.norm(np.gradient(x, axis=0) * fs, axis=1)
        P.append(x[v < vthr])
    P = np.vstack(P)
    H = P[:, [0, 2]] - P[:, [0, 2]].mean(axis=0)
    U = np.linalg.svd(H, full_matrices=False)[2]        # 踵が動いた主な向き（水平面内）
    S = H @ U.T
    use = [k for k in range(2) if np.subtract(*np.percentile(S[:, k], [95, 5])) >= min_spread]
    if not use:
        return {'a': np.nan, 'b': np.nan, 'c': np.median(P[:, 1]), 'tilt_deg': np.nan,
                'rms_m': np.std(P[:, 1]), 'n': len(P)}
    A = np.column_stack([S[:, use], np.ones(len(P))])
    coef = np.linalg.lstsq(A, P[:, 1], rcond=None)[0]
    g = np.zeros(2)
    g[use] = coef[:-1]
    a, b = g @ U                                         # 傾きを x，z の向きに戻す
    c = coef[-1] - a * P[:, 0].mean() - b * P[:, 2].mean()
    rms = np.sqrt(np.mean((P[:, 1] - (a * P[:, 0] + b * P[:, 2] + c)) ** 2))
    return {'a': a, 'b': b, 'c': c, 'tilt_deg': np.degrees(np.arctan(np.hypot(a, b))), 'rms_m': rms, 'n': len(P)}


def level_rotation(plane):
    """床面の法線を鉛直（Y）に重ねる回転行列．傾きを求められなかったときは回さない"""
    if not np.isfinite(plane['a']):
        return np.eye(3)
    n = np.array([-plane['a'], 1.0, -plane['b']])
    n = n / np.linalg.norm(n)
    v, c = np.cross(n, UP), n @ UP
    s = np.linalg.norm(v)
    if s < 1e-12:
        return np.eye(3)
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx * ((1 - c) / s ** 2)


def level_trc(trc, plane):
    """床が水平になるように，すべてのマーカーを原点のまわりに回す"""
    Rm = level_rotation(plane)
    return {**trc, 'markers': {k: v @ Rm.T for k, v in trc['markers'].items()}}


def level_mot(mot, plane):
    """骨盤の位置（pelvis_tx，pelvis_ty，pelvis_tz）を level_trc と同じように回す．
    関節角度は変わらない．骨盤の向き（pelvis_tilt など）は直さないので，傾いた分（数度）ずれたまま"""
    out = mot.copy()
    cols = ['pelvis_tx', 'pelvis_ty', 'pelvis_tz']
    out[cols] = mot[cols].values @ level_rotation(plane).T
    return out

def gait_events_zeni(trc, side):
    """踵接地（hs）とつま先離地（to）の時刻を求める（Zeni ら 2008 の座標法．）．
    side は 'r'（右）か 'l'（左）．
    踵接地：踵マーカーが骨盤（左右の上後腸骨棘の中点）より最も前に出た時刻．
    つま先離地：つま先マーカーが骨盤より最も後ろに下がった時刻．
    前後方向は両足の「踵→つま先」の向きの平均（水平面内）．"""
    m = trc['markers']
    p = 'r' if side.lower().startswith('r') else 'L'
    turn = np.ptp(heading(trc))
    if turn > 45:
        warnings.warn('試行の中で体の向きが %.0f° 変わっています（方向転換）．前後の向きを1つに決める方法なので，'
                      'straight_passes で直進の区間に分けてから使ってください' % turn)
    sacrum = (m['r.PSIS_study'] + m['L.PSIS_study']) / 2
    fwd = np.vstack([m['r_toe_study'] - m['r_calc_study'], m['L_toe_study'] - m['L_calc_study']]).mean(axis=0)
    fwd[1] = 0.0                                     # 鉛直成分を除いて水平面内の向きにする
    fwd = fwd / np.linalg.norm(fwd)
    ap = {'heel': (m[p + '_calc_study'] - sacrum) @ fwd,
          'toe': (m[p + '_toe_study'] - sacrum) @ fwd}
    w = int(round(0.4 * trc['rate']))                # 前後 0.4 s の中で最も前（後ろ）の点だけを採る
    hs = trc['time'][_local_maxima(ap['heel'], w)]
    to = trc['time'][_local_maxima(-ap['toe'], w)]
    return hs, to, ap


def _local_maxima(x, w):
    """山の頂点のうち，平均より大きく，前後 w サンプルの中で最大のもの"""
    n, mu = len(x), np.nanmean(x)
    return np.array([i for i in range(1, n - 1)
                     if x[i] > x[i - 1] and x[i] >= x[i + 1] and x[i] > mu
                     and x[i] == np.max(x[max(0, i - w):min(n, i + w + 1)])], dtype=int)


def heading(trc, fc=1.0):
    """体の向き（度）：上後腸骨棘の中点 → 上前腸骨棘の中点の向きを水平面で測り，1 Hz で平滑化したもの
    （歩くときの骨盤の回旋を消すため）．値そのものより変化（方向転換で 180° など）を見る．"""
    m = trc['markers']
    h = (m['r.ASIS_study'] + m['L.ASIS_study']) / 2 - (m['r.PSIS_study'] + m['L.PSIS_study']) / 2
    return np.degrees(lowpass(np.unwrap(np.arctan2(h[:, 2], h[:, 0])), trc['rate'], fc))


def straight_passes(trc, tol_deg=35.0, turn_rate=50.0, min_speed=0.3, min_dur=1.0, min_step=0.3):
    """まっすぐ歩いている区間（直進）を [(開始の番号, 終了の番号の次), ...] で返す．
    骨盤の中心が進む向き（1 Hz で平滑化）を使う．進む速さが min_speed m/s 以上で，向きの変わる速さが
    turn_rate 度/s 未満，区間の初めから tol_deg 以内の向きが min_dur 秒以上続き，踵が骨盤に対して
    min_step m 以上前後に動いている（歩いている）区間．
    TUG の帰り道のようにゆるく曲がる区間（向きの変化 20〜40 度/s）は含め，方向転換（100 度/s 超）で区切る．
    その場で歩く試行（トレッドミル）は，体の向き（heading）が 45° 以上変わらなければ全体を1区間とする"""
    m, fs = trc['markers'], trc['rate']
    sacrum = (m['r.PSIS_study'] + m['L.PSIS_study']) / 2
    pc = lowpass(pelvis_center(trc)[:, [0, 2]], fs, 1.0)
    vel = np.gradient(pc, axis=0) * fs
    speed = np.hypot(vel[:, 0], vel[:, 1])

    def walking(a, b, d):
        fwd = np.array([np.cos(d), 0.0, np.sin(d)])
        heel = (m['r_calc_study'][a:b] - sacrum[a:b]) @ fwd
        return np.subtract(*np.percentile(heel, [95, 5])) >= min_step

    if np.ptp(pc, axis=0).max() < 1.0:               # その場で歩いている
        psi = heading(trc)
        n = len(psi)
        return [(0, n)] if np.ptp(psi) < 45 and walking(0, n, np.radians(np.median(psi))) else []
    ang = np.degrees(np.unwrap(np.arctan2(vel[:, 1], vel[:, 0])))
    ok = (speed >= min_speed) & (np.abs(np.gradient(ang) * fs) < turn_rate)
    out, i, n = [], 0, len(ang)
    while i < n:
        if not ok[i]:
            i += 1
            continue
        j = i
        while j + 1 < n and ok[j + 1] and abs(ang[j + 1] - ang[i]) <= tol_deg:
            j += 1
        a, b = i, j + 1
        if (b - a) / fs >= min_dur and walking(a, b, np.radians(np.median(ang[a:b]))):
            out.append((a, b))
        i = b
    return out


def trial_slice(trc, a, b, mot=None):
    """試行の一部（番号 a から b の手前まで）を取り出す．mot を渡すと同じ時間の .mot も返す"""
    sub = {**trc, 'time': trc['time'][a:b], 'markers': {k: v[a:b] for k, v in trc['markers'].items()}}
    if mot is None:
        return sub
    t0, t1 = trc['time'][a], trc['time'][b - 1]
    return sub, mot[(mot['time'] >= t0 - 1e-6) & (mot['time'] <= t1 + 1e-6)].reset_index(drop=True)


def gait_events_by_pass(trc, **kw):
    """直進の区間ごとに gait_events_zeni を実行する（方向転換をはさむ試行や TUG のため）．
    戻り値：区間ごとの辞書のリスト．{'t0', 't1', 'r': (hs, to), 'l': (hs, to)}．
    歩行周期は各直進区間の中だけで作る"""
    out = []
    for a, b in straight_passes(trc, **kw):
        sub = trial_slice(trc, a, b)
        ev = {s: gait_events_zeni(sub, s)[:2] for s in ('r', 'l')}
        out.append({'t0': trc['time'][a], 't1': trc['time'][b - 1], **ev})
    return out


def _crossing(psi, start, a, b, frac):
    """start 以降で，向き psi が a から b への変化の frac（0〜1）に初めて達した番号（なければ None）"""
    i = np.flatnonzero((psi[start:] - a) * np.sign(b - a) >= frac * abs(b - a))
    return start + int(i[0]) if len(i) else None


def segment_tug(mot, trc, fc=4.0, v_on=0.1, lean_thr=3.0, lo=0.2, hi=0.8):
    """Timed Up and Go（椅子から立ち上がり，先の目印で向きを変えて戻り，座る）を区切る．
    床が傾いて再構成されたデータは，先に level_mot・level_trc で直してから渡す．
    - 座位の高さ：骨盤の高さ（pelvis_ty）の最初と最後の 0.3 s の中央値．立位の高さ：全体の 75 パーセンタイル
    - 立ち上がり：骨盤が座位と立位の中間を初めて超えた時点から前後にたどり，上向きの速さが v_on m/s を
      超えている区間を「離殿 → 立位」とする．その前で体幹の前傾の角速度が lean_thr 度/s を超え続けた
      最初の時点を「動き始め」とする
    - 着座：骨盤が最後に中間より上にあった時点から前後にたどり，下向きの速さが v_on m/s を超えている区間を
      「座り始め → 着座」とする
    - 方向転換：体の向き（heading，0.5 Hz で平滑化）が，行きの向きから帰りの向き（行き ± 180°）へ，
      帰りの向きから最後（座位）の向きへ変わる途中の，変化の 20 % → 80 % の区間．
      向きのゆれ（骨盤の回旋や計測のゆれ）に左右されにくいように，変わり始め・終わりの 20 % は含めない
    戻り値：時刻（s）と時間（_s）の辞書．試行の最初や最後に座位がないとき（撮り始めが遅い，切り詰められた）は nan．
    tug_s は「動き始め → 着座」．合図から計る本来の TUG とは始まりが違う"""
    t = mot['time'].values
    fs = 1.0 / np.median(np.diff(t))
    p = lowpass(mot['pelvis_ty'].values, fs, fc)
    v = np.gradient(p, t)
    k = max(2, int(0.3 * fs))
    seat0, seat1, stand = np.median(p[:k]), np.median(p[-k:]), np.percentile(p, 75)
    keys = ['lean_onset', 'seat_off', 'standing', 'turn1_start', 'turn1_end', 'turn2_start', 'turn2_end',
            'sit_start', 'seated']
    r = dict.fromkeys(keys, np.nan)
    if stand - seat0 > 0.15:
        iu = int(np.argmax(p > (seat0 + stand) / 2))
        i0, i1 = iu, iu
        while i0 > 0 and v[i0] >= v_on:
            i0 -= 1
        while i1 < len(t) - 1 and v[i1] >= v_on:
            i1 += 1
        if i0 > 0:                                   # 立ち上がりの途中から記録が始まっていない
            lean = trunk_lean(trc)
            w = np.gradient(lowpass(np.interp(t, lean['time'], lean['sagittal']), fs, fc), t)
            il = i0
            while il > 0 and w[il - 1] > lean_thr:
                il -= 1
            r.update(lean_onset=t[il], seat_off=t[i0], standing=t[i1])
    if stand - seat1 > 0.15:
        idn = len(p) - 1 - int(np.argmax(p[::-1] > (seat1 + stand) / 2))
        j0, j1 = idn, idn
        while j0 > 0 and v[j0] <= -v_on:
            j0 -= 1
        while j1 < len(t) - 1 and v[j1] <= -v_on:
            j1 += 1
        if j1 < len(t) - 1:                          # 座り終わる前に記録が終わっていない
            r.update(sit_start=t[j0], seated=t[j1])
    # 方向転換（行き → 帰り，帰り → 最後の向き）
    psi, tt = heading(trc, 0.5), trc['time']
    q = max(2, int(0.5 * trc['rate']))
    psi0, psi_end = np.median(psi[:q]), np.median(psi[-q:])
    big = np.flatnonzero(np.abs(psi - psi0) > 135)
    n_turns = 0
    if len(big):
        psi1 = psi0 + 180.0 * np.sign(psi[big[0]] - psi0)
        a = _crossing(psi, 0, psi0, psi1, lo)
        b = _crossing(psi, a, psi0, psi1, hi) if a is not None else None
        if b is not None:
            r.update(turn1_start=tt[a], turn1_end=tt[b])
            n_turns = 1
            if abs(psi_end - psi1) > 90:
                # 帰りの向きに最も近づいた所から，2回目を探す（2回目が逆回りでも1回目の終わりと重ねない）
                e = _crossing(psi, b, psi0, psi1, 0.95)
                if e is None:
                    e = b + int(np.argmax((psi[b:] - psi0) * np.sign(psi1 - psi0)))
                c = _crossing(psi, e, psi1, psi_end, lo)
                d = _crossing(psi, c, psi1, psi_end, hi) if c is not None else None
                if d is not None:
                    r.update(turn2_start=tt[c], turn2_end=tt[d])
                    n_turns = 2
    r.update(n_turns=n_turns, tug_s=r['seated'] - r['lean_onset'], rise_s=r['standing'] - r['lean_onset'],
             walk_out_s=r['turn1_start'] - r['standing'], turn1_s=r['turn1_end'] - r['turn1_start'],
             walk_back_s=r['turn2_start'] - r['turn1_end'], turn_sit_s=r['seated'] - r['turn2_start'])
    return r

In [ ]:
# 関数：データの読み込み，歩行周期の表と選び方，区間の検査
import re

import yaml

# 展開するファイル（セッションのフォルダからの相対パス）．元の動画（.mov），keypoint の pickle，Geometry は展開しない
KEEP = re.compile(r'^(sessionMetadata\.yaml|MarkerData/[^/]+\.trc|OpenSimData/Kinematics/[^/]+\.mot'
                  r'|OpenSimData/Model/(?:[^/]+/)?[^/]+\.osim)$')


def extract_session(zip_path, dest):
    """OpenCap の zip から，このノートブックで使うファイルだけを dest に展開する（dest にあったものは消す）．
    zip の中のセッションのフォルダ（OpenCapData_<ID>/ など．なくてもよい）は外す．動画は展開しない．
    戻り値：展開したファイルの数"""
    with zipfile.ZipFile(zip_path) as z:
        names = [i.filename for i in z.infolist() if not i.is_dir()]
        hit = next(('/' + n for n in names if '/MarkerData/' in '/' + n), None)
        if hit is None:
            raise ValueError('MarkerData（.trc）が見つかりません．OpenCap からダウンロードした zip か確かめてください')
        prefix = hit[:hit.index('/MarkerData/')].lstrip('/')
        prefix = prefix + '/' if prefix else ''
        shutil.rmtree(dest, ignore_errors=True)
        n = 0
        for name in names:
            if not name.startswith(prefix):
                continue
            rel = name[len(prefix):]
            if '..' in rel.split('/') or not KEEP.match(rel):
                continue
            out = os.path.join(dest, *rel.split('/'))
            os.makedirs(os.path.dirname(out), exist_ok=True)
            with z.open(name) as src, open(out, 'wb') as f:
                shutil.copyfileobj(src, f)
            n += 1
    return n


def trc_header(path):
    """.trc の見出しから（コマ数, fps）"""
    with open(path, encoding='utf-8', errors='replace') as f:
        lines = [f.readline() for _ in range(3)]
    keys, vals = lines[1].rstrip('\n').split('\t'), lines[2].rstrip('\n').split('\t')
    return int(vals[keys.index('NumFrames')]), float(vals[keys.index('DataRate')])


def session_meta(folder):
    """sessionMetadata.yaml の中身（なければ空）"""
    p = os.path.join(folder, 'sessionMetadata.yaml')
    if not os.path.exists(p):
        return {}
    with open(p, encoding='utf-8') as f:
        return yaml.safe_load(f) or {}


def session_info(folder):
    """試行の一覧，身長・体重，モデル名を返す．"""
    meta = session_meta(folder)
    rows = []
    for trc in sorted(glob.glob(os.path.join(folder, 'MarkerData', '*.trc'))):
        name = os.path.splitext(os.path.basename(trc))[0]
        n, fps = trc_header(trc)
        rows.append({'trial': name, 'duration_s': round((n - 1) / fps, 2), 'fps': fps,
                     'mot': os.path.exists(os.path.join(folder, 'OpenSimData', 'Kinematics', name + '.mot'))})
    return {'trials': pd.DataFrame(rows, columns=['trial', 'duration_s', 'fps', 'mot']),
            'height_m': meta.get('height_m'), 'mass_kg': meta.get('mass_kg'),
            'model': meta.get('openSimModel', 'LaiUhlrich2022')}


def model_file(folder, trial, model='LaiUhlrich2022'):
    """スケーリングしたモデル．1台版は試行ごと（OpenSimData/Model/<試行>/），2台以上はセッションに1つ"""
    for p in (os.path.join(folder, 'OpenSimData', 'Model', trial, model + '_scaled.osim'),
              os.path.join(folder, 'OpenSimData', 'Model', model + '_scaled.osim')):
        if os.path.exists(p):
            return p
    raise FileNotFoundError('スケーリングしたモデル（%s_scaled.osim）が見つかりません' % model)

TUG_PHASES = (('rise', 'lean_onset', 'standing'), ('turn', 'turn1_start', 'turn1_end'),
              ('turn_back', 'turn2_start', 'turn2_end'), ('sit', 'sit_start', 'seated'))


def tug_phases(mot, trc):
    """TUG の区切り {名前: (開始, 終了)}．rise（立ち上がり），turn（方向転換），turn_back（戻って座る前の向き直し），sit（着座）．
    床の傾きを直してから segment_tug に渡す．区切れないもの（座っていない，向きを変えていない）は入れない"""
    try:
        pl = floor_plane(trc)
        seg = segment_tug(level_mot(mot, pl), level_trc(trc, pl))
    except Exception:  # noqa: BLE001  歩行だけの試行などで区切れないとき
        return {}
    return {k: (float(seg[a]), float(seg[b])) for k, a, b in TUG_PHASES
            if np.isfinite(seg.get(a, np.nan)) and np.isfinite(seg.get(b, np.nan))}


def gait_analysis(trc, mot):
    """直進区間ごとの歩行イベントと TUG の区切りを返す．"""
    with warnings.catch_warnings():   # 直進の区間に分けたあとなので，gait_events_zeni の「方向転換を含む」の警告は出さない
        warnings.filterwarnings('ignore', message='試行の中で体の向き')
        passes = gait_events_by_pass(trc)
    phases = tug_phases(mot, trc)
    return passes, phases


CYCLE_COLUMNS = ['pass', 'side', 'start_s', 'end_s', 'duration_s', 'ok', 'note']


def turn_onset(trc, frac=0.05, start=None):
    """方向転換の始まり：骨盤の向き（0.5 Hz で平滑化）が，行きの向き（最初の 0.5 s の中央値）から帰りの向き
    （行き ± 180° から 30° 以内の時点の中央値）へ変わる途中の frac（既定 5 %）に，start 秒より後で初めて達した時刻．
    向きを変えない試行は nan"""
    psi, t = heading(trc, 0.5), trc['time']
    q = max(2, int(0.5 * trc['rate']))
    out = float(np.median(psi[:q]))
    big = np.flatnonzero(np.abs(psi - out) > 135)
    if not len(big):
        return np.nan
    back0 = out + 180.0 * np.sign(psi[big[0]] - out)
    near = np.abs(psi - back0) < 30
    if not near.any():                                   # 帰りの向きになっていない（向きの変化が足りない）
        return np.nan
    back = float(np.median(psi[near]))
    i = np.flatnonzero((t >= (start or 0.0)) & ((psi - out) * np.sign(back - out) >= frac * abs(back - out)))
    return float(t[i[0]]) if len(i) else np.nan


def walkout_cycles(passes, phases, onset, max_cycle=1.6):
    """往路の歩行周期の表：立ち上がりの終わり（phases の rise．なければ 0 秒）より後に始まる，同じ足の踵接地から次の踵接地まで
    （直進の区間の中．max_cycle 秒未満）．方向転換の始まり onset より後に始まる直進の区間（帰り）は入れない．
    ok：方向転換の始まりまでに終わる．方向転換の始まりが分からない（nan）ときは，帰りの区間を往路と取り違えないよう，どれも使わない．
    並べる順：右の周期，左の周期（それぞれ時刻の順）"""
    rise_end = (phases or {}).get('rise', (0.0, 0.0))[1]
    lim = onset if np.isfinite(onset) else np.inf
    rows = []
    for side, key in (('R', 'r'), ('L', 'l')):
        for i, p in enumerate(passes):
            if p['t0'] >= lim or p['t1'] <= rise_end:
                continue
            hs = np.sort(np.asarray(p[key][0], dtype=float))
            for a, b in zip(hs[:-1], hs[1:]):
                if a < rise_end or b - a >= max_cycle:
                    continue
                if not np.isfinite(onset):
                    ok, note = False, '方向転換の始まりが見つからない（往路と帰りを分けられない）'
                else:
                    ok = bool(b <= lim)
                    note = '' if ok else '終わりが方向転換の始まり（%.2f 秒）を過ぎる' % onset
                rows.append({'pass': i + 1, 'side': side, 'start_s': float(a), 'end_s': float(b), 'duration_s': float(b - a),
                             'ok': ok, 'note': note})
    df = pd.DataFrame(rows, columns=CYCLE_COLUMNS)
    df.index = pd.RangeIndex(1, len(df) + 1, name='no')
    return df


def choose_cycle(cycles):
    """使う歩行周期：ok の周期のうち，終わりがいちばん早いもの（方向転換から遠い）．戻り値：{'no', 'side', 't0', 't1'}"""
    ok = cycles[cycles['ok']]
    if not len(ok):
        raise ValueError('往路に，方向転換の始まりまでに終わる歩行周期がありません')
    k = ok['end_s'].idxmin()
    r = ok.loc[k]
    return {'no': int(k), 'side': r['side'], 't0': float(r['start_s']), 't1': float(r['end_s'])}


def steady_cycles(passes, phases=None, max_cycle=1.6):
    """ふつうの歩行の歩行周期の表：直進の区間ごと・足ごとに，踵接地から同じ足の次の踵接地まで（max_cycle 秒未満）．
    ok：その区間のその足の最初と最後の周期でなく（歩き始め・歩き終わりで速さが変わる），TUG の区切りにかからない．
    ただし，その足の周期が 2 つ以下（外すと何も残らない）なら最初と最後も ok にして，注を付ける（OpenCap の撮影の範囲では多い）．
    並べる順：区間の順に，右の周期，左の周期（それぞれ時刻の順）"""
    rows = []
    for i, p in enumerate(passes):
        for side, key in (('R', 'r'), ('L', 'l')):
            hs = np.sort(np.asarray(p[key][0], dtype=float))
            cyc = [(a, b) for a, b in zip(hs[:-1], hs[1:]) if b - a < max_cycle]
            for j, (a, b) in enumerate(cyc):
                notes, short = [], len(cyc) <= 2
                if short:
                    notes.append('区間が短い（歩き始め・歩き終わりを含むことがある）')
                else:
                    if j == 0:
                        notes.append('区間の最初の周期（歩き始め）')
                    if j == len(cyc) - 1:
                        notes.append('区間の最後の周期（歩き終わり）')
                bad = any(a < y and b > x for x, y in (phases or {}).values())
                if bad:
                    notes.append('TUG の区切りにかかる')
                ok = not bad and (short or 0 < j < len(cyc) - 1)
                rows.append({'pass': i + 1, 'side': side, 'start_s': float(a), 'end_s': float(b), 'duration_s': float(b - a),
                             'ok': ok, 'note': '，'.join(notes)})
    df = pd.DataFrame(rows, columns=CYCLE_COLUMNS)
    df.index = pd.RangeIndex(1, len(df) + 1, name='no')
    return df


def choose_steady(cycles, passes):
    """ふつうの歩行で使う歩行周期：ok の周期のある直進の区間のうちいちばん長い区間で，中点が区間の中点にいちばん近い周期．
    戻り値：{'no', 'side', 't0', 't1'}"""
    ok = cycles[cycles['ok']]
    if not len(ok):
        raise ValueError('直進の区間の途中（最初と最後を除く）に歩行周期がありません．WINDOW で区間を指定してください')
    longest = max(set(ok['pass']), key=lambda n: passes[n - 1]['t1'] - passes[n - 1]['t0'])
    p = passes[longest - 1]
    cand = ok[ok['pass'] == longest]
    k = ((cand['start_s'] + cand['end_s']) / 2 - (p['t0'] + p['t1']) / 2).abs().idxmin()
    r = ok.loc[k]
    return {'no': int(k), 'side': r['side'], 't0': float(r['start_s']), 't1': float(r['end_s'])}

def check_window(t0, t1, times, passes=None, phases=None, tilt_deg=np.nan):
    """区間の検査．開始と終了をいちばん近いコマに合わせ，止めるもの（errors）と注意（notes）を返す．
    passes は直進の区間の一覧（None なら直進の区間は確かめない．空なら「見つからない」と注意する）"""
    times = np.asarray(times, dtype=float)
    errors, notes = [], []
    lo, hi = float(times[0]), float(times[-1])
    half = 0.5 * float(np.median(np.diff(times)))
    if t0 < lo - half or t1 > hi + half:
        errors.append('試行の範囲（%.3f〜%.3f 秒）の外です' % (lo, hi))
    k0, k1 = int(np.argmin(np.abs(times - t0))), int(np.argmin(np.abs(times - t1)))
    t0, t1 = float(times[k0]), float(times[k1])
    if t1 <= t0:
        errors.append('終了が開始と同じか，開始より前です')
    elif t1 - t0 < 0.3:
        errors.append('0.3 秒より短い区間は解けません')
    elif t1 - t0 > 1.6:
        notes.append('1.6 秒より長い区間です．解くのに時間がかかります（1歩は 0.5〜0.7 秒ほど）')
    if not errors:
        if t0 - lo < 0.1 or hi - t1 < 0.1:
            notes.append('試行の始まりか終わりから 0.1 秒以内です．OpenSimAD は試行全体にフィルタをかけるので，端の近くは信頼しにくくなります')
        if passes is not None and len(passes) == 0:
            notes.append('この試行には直進の区間が見つかりません（歩行でない試行かもしれません）．このノートブックの OpenSimAD の設定は歩行用です')
        elif passes and not any(p['t0'] <= t0 and t1 <= p['t1'] for p in passes):
            notes.append('直進の区間の外を含みます（方向転換などで，歩行の設定に合わないことがあります）')
        if phases and any(t0 < b and t1 > a for a, b in phases.values()):
            notes.append('TUG の立ち上がり・方向転換・着座の区間にかかっています')
    if np.isfinite(tilt_deg) and tilt_deg > 2:
        notes.append('床が %.1f° 傾いて再構成されています．解く前に水平にしますが，キャリブレーションが悪い撮影なので，'
                     'ほかの誤差も大きいかもしれません' % tilt_deg)
    return {'t0': t0, 't1': t1, 'k0': k0, 'k1': k1, 'errors': errors, 'notes': notes}

In [ ]:
# 関数：結果の確認，OpenSim の環境とのやりとり
JOINTS = ['hip_flexion', 'knee_angle', 'ankle_angle']
JOINTS6 = [j + s for s in ('_r', '_l') for j in JOINTS]
JOINT_JA = {'hip_flexion': '股関節屈曲', 'knee_angle': '膝関節屈曲', 'ankle_angle': '足関節背屈'}
LIMITS = {'peak_bw': 2.0, 'mean_bw': (0.8, 1.2), 'dev_deg': 8.0}   # 仮の目安


def joint_ja(name):
    return ('右' if name.endswith('_r') else '左') + JOINT_JA[name[:-2]]


def read_result(session, trial, case, mass_kg, input_mot):
    """OpenSimAD の結果（床反力，関節モーメント，解の関節角度）と，確認に使う値．ファイルがなければ None．
    床反力は体重比，ずれは入力（OpenCap）の角度との RMSE（度，6関節）"""
    d = os.path.join(session, 'OpenSimData', 'Dynamics', trial)
    paths = {k: os.path.join(d, '%s_%s_%s.mot' % (k, trial, case))
             for k in ('GRF_resultant', 'kinetics', 'kinematics_activations')}
    if not all(os.path.exists(p) for p in paths.values()):
        return None
    grf, mom, kin = (read_mot(paths[k]) for k in ('GRF_resultant', 'kinetics', 'kinematics_activations'))
    ik = read_mot(input_mot)
    bw = float(mass_kg) * 9.81
    fr, fl = grf['ground_force_right_vy'].values / bw, grf['ground_force_left_vy'].values / bw
    dev = {j: float(np.sqrt(np.mean((np.interp(kin['time'], ik['time'], ik[j]) - kin[j]) ** 2))) for j in JOINTS6}
    jmax = max(dev, key=dev.get)
    return {'grf': grf, 'mom': mom, 'kin': kin, 'ik': ik, 'bw': bw, 'mass_kg': float(mass_kg),
            'peak_r': float(fr.max()), 'peak_l': float(fl.max()), 'mean_bw': float(np.mean(fr + fl)),
            'dev': dev, 'dev_max': dev[jmax], 'dev_joint': joint_ja(jmax)}


def verdict(status, res):
    """結果の確認．戻り値：（通ったか，当たった理由のリスト）．目安は LIMITS（仮のもの）"""
    if status != 'Solve_Succeeded':
        return False, ['収束しませんでした（%s）' % status]
    if res is None:
        return False, ['結果のファイルが見つかりません']
    reasons = []
    peak = max(res['peak_r'], res['peak_l'])
    if peak > LIMITS['peak_bw']:
        reasons.append('鉛直床反力のピークが体重の %.1f 倍です（目安は 2 倍以下）．歩行としては大きすぎます．'
                       '足が床にめり込んだ入力を，強い力で押し返した解のおそれがあります' % peak)
    lo, hi = LIMITS['mean_bw']
    if not lo <= res['mean_bw'] <= hi:
        reasons.append('1歩の間の鉛直床反力の平均（左右の和）が体重の %.2f 倍です（目安は 0.8〜1.2 倍）．'
                       '体重を支えきれていない（または支えすぎている）ので，入力の動きと床との位置関係が合っていないおそれがあります'
                       % res['mean_bw'])
    if res['dev_max'] >= LIMITS['dev_deg']:
        reasons.append('入力の角度からのずれ（RMSE）が%sで %.1f° です（目安は 8° 未満）．OpenSimAD が入力の動きを大きく直しました．'
                       '入力のままでは物理的に成り立たない（例：足が床にめり込む）おそれがあります' % (res['dev_joint'], res['dev_max']))
    return not reasons, reasons


def check_body(mass_kg, height_m, where='1 節の入力のセル'):
    """体重（kg）と身長（m）を確かめる．ないか，ありえない値（打ち間違い）なら ValueError．where：直す場所（文に入れる）．
    体重は床反力を体重比に直すのに使うので，打ち間違うと，どの1歩も確認に当たってしまう"""
    if not mass_kg or not height_m:
        raise ValueError('体重と身長が要ります．%sの MASS_KG（kg）と HEIGHT_M（m）に入れて，そこから実行し直してください' % where)
    if not 20 <= float(mass_kg) <= 250 or not 1.0 <= float(height_m) <= 2.3:
        raise ValueError('体重 %g kg・身長 %g m は打ち間違いのようです（体重は kg，身長は m で．例 65 と 1.72）．'
                         '%sを直して，そこから実行し直してください' % (float(mass_kg), float(height_m), where))


# 筋活動の図に出す右脚の筋（LaiUhlrich2022 の名前）
ACT_MUSCLES = ['glmax1_r', 'glmed1_r', 'recfem_r', 'vasmed_r', 'bflh_r', 'gasmed_r', 'soleus_r', 'tibant_r']
ACT_NAMES = {'glmax1_r': 'gluteus maximus', 'glmed1_r': 'gluteus medius', 'recfem_r': 'rectus femoris',
             'vasmed_r': 'vastus medialis', 'bflh_r': 'biceps femoris (long head)', 'gasmed_r': 'gastrocnemius medialis',
             'soleus_r': 'soleus', 'tibant_r': 'tibialis anterior'}


def pack_results(folder, out_zip, top):
    """folder の中のファイルを，top/ の下に置いた zip にする"""
    with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
        for r, _, fs in sorted(os.walk(folder)):
            for f in sorted(fs):
                p = os.path.join(r, f)
                z.write(p, top + '/' + os.path.relpath(p, folder).replace(os.sep, '/'))
    return out_zip


def readme_check_line(info):
    """README の「OpenSim で読み込めるか」の行．確かめられなかったときは，性質を言い切らない"""
    head = '- OpenSim で読み込めるか（OpenSim 4.5 の API で確かめた）：'
    if not info.get('opensim_ok', True):
        return head + '問題があった（runner.log の export のところを見る）'
    line = head + '確かめた：座標と筋の名前がモデルと合う，床反力の列と踵骨の名前がそろう'
    ida = info.get('id_agreement')
    if ida:
        line += ('．接触球の力を切ったモデルで，筋を除いて逆動力学（InverseDynamicsTool）を計算し直すと，OpenSimAD の関節モーメントと一致した'
                 '（股・膝・足関節の屈伸の左右で，相関 %.2f 以上，差の RMS は振れ幅の %.1f %% 以下）' % (ida['min_corr'], 100 * ida['max_rms_rel']))
    return line


def results_readme(info):
    """区間，設定，所要時間，確認値，出力ファイルを README にまとめる．"""
    f, c = info['files'], info['checks']
    kind = '筋'
    window_line = info['window_line']
    data_notes = info['data_notes']
    lines = [info['title'], '', info['data_line'],
             window_line,
             '- 設定：%s駆動，メッシュ %d（1 秒あたりの区間の数），opencap-processing 72b5416，OpenSim 4.5，CasADi 3.7.2．%s'
             % (kind, info['mesh'], info.get('filters', '追う量のフィルタは opencap-processing の歩行の設定（walking）の既定')),
             '- 求解：%s，%s 回' % (info['status'], info['iter']),
             '- 確認：鉛直床反力のピーク 右 %.2f・左 %.2f 体重比，区間の平均（左右の和）%.2f 体重比，入力の角度からのずれ（RMSE）の最大 %.1f°（%s）'
             % (c['peak_r'], c['peak_l'], c['mean_bw'], c['dev_max'], c['dev_joint']),
             readme_check_line(info), '',
             '## 所要時間（%s）' % info['environment'], '', '| 段階 | 秒 | 分 |', '|---|---|---|']
    lines += ['| %s | %.0f | %.1f |' % (k, v, v / 60) for k, v in info['times'].items()]
    descriptions = {'model': '接触球の力を無効にしたモデル', 'motion': '関節角度（度）と骨盤の並進（m）',
                    'motion_activations': '関節角度と筋活動', 'GRF': '床反力（N），作用点（m），自由モーメント（Nm）',
                    'external_loads': '床反力を踵骨に加える設定', 'id_setup': '筋を除く逆動力学の設定',
                    'kinetics': '関節モーメント（Nm）と骨盤の力（N）', 'model_original': '接触球の力が有効な元モデル'}
    lines += ['', '## ファイル', '', '| ファイル | 中身 |', '|---|---|']
    lines += ['| `%s` | %s |' % (path, descriptions.get(key, key)) for key, path in f.items()]
    lines += ['| `opensimad_original/` | OpenSimAD の元出力 |']
    lines += ['| `%s` | 結果の図 |' % x for x in info.get('figures', [])]
    lines += ['',
              '## 注意', '',
              '- 逆動力学は，接触球の力を切ったモデルで，筋を除いて計算する．接触球の力が有効なモデルに ExternalLoads を足すと床反力が二重に入り，'
              '筋を除かないと既定の状態の筋の力も入る．OpenSim の逆動力学は関節角度を微分して加速度を求めるので，OpenSimAD の値と少し違う'
              '（骨盤の残差も 0 にならない）．',
              '- 床反力は N のまま（体重比にするには，体重 × 9.81 で割る）．',
              '- 区間の端（始まりと終わり）は解釈に使わない．メッシュが粗い（%d）ので，曲線は滑らかではない．' % info['mesh']]
    lines += list(data_notes)
    return '\n'.join(lines) + '\n'

ROOT = os.environ.get('OSAD_ROOT', '/opt/osad')   # OpenSim の環境と作業のフォルダ
IPOPT_LINE = re.compile(r'^\s*(\d+)r?\s+([-+]?\d\.\d+e[-+]\d+)\s+([-+]?\d\.\d+e[-+]\d+)')


def write_tools(files):
    """OpenSim の環境で使うファイル（osad_runner.py，oc_research_lite.py）を ROOT に書く"""
    os.makedirs(ROOT, exist_ok=True)
    for name, text in files.items():
        with open(os.path.join(ROOT, name), 'w', encoding='utf-8') as f:
            f.write(text)


def reset_session(sid):
    """opencap-processing の作業のフォルダ（前処理の結果を含む）を消す．別の zip を読み込んだときに使う"""
    shutil.rmtree(os.path.join(ROOT, 'Data', sid), ignore_errors=True)


def prepare_session(src, sid, trial, mass_kg, height_m, model='LaiUhlrich2022'):
    """opencap-processing が読む形（ROOT/Data/<sid>/）にファイルを置く．sessionMetadata.yaml には体重・身長を書く．
    戻り値：セッションのフォルダ"""
    check_body(mass_kg, height_m)
    dst = os.path.join(ROOT, 'Data', sid)
    for sub in ('OpenSimData/Kinematics', 'MarkerData'):
        os.makedirs(os.path.join(dst, sub), exist_ok=True)
    shutil.copy(os.path.join(src, 'OpenSimData', 'Kinematics', trial + '.mot'), os.path.join(dst, 'OpenSimData', 'Kinematics'))
    shutil.copy(os.path.join(src, 'MarkerData', trial + '.trc'), os.path.join(dst, 'MarkerData'))
    mf = model_file(src, trial, model)
    out = os.path.join(dst, 'OpenSimData', 'Model', os.path.relpath(mf, os.path.join(src, 'OpenSimData', 'Model')))
    os.makedirs(os.path.dirname(out), exist_ok=True)
    if not os.path.exists(out):
        shutil.copy(mf, out)
    meta = dict(session_meta(src))
    meta.update(mass_kg=float(mass_kg), height_m=float(height_m), openSimModel=model)
    with open(os.path.join(dst, 'sessionMetadata.yaml'), 'w', encoding='utf-8') as f:
        yaml.safe_dump(meta, f, allow_unicode=True, sort_keys=False)
    return dst


def run_osad(cmd, args):
    """osad_runner.py を OpenSim の環境で動かす．[osad] の行と IPOPT の進捗（約30秒間隔）を表示し，すべての行を ROOT/runner.log に書く．
    セルを中断したときは，裏で解き続けないように子のプロセスを止める．戻り値：osad_runner.py が最後に出した RESULT の辞書"""
    env = dict(os.environ, PATH=os.path.join(ROOT, 'env', 'bin') + os.pathsep + os.environ.get('PATH', ''))
    cmdline = [os.path.join(ROOT, 'env', 'bin', 'python'), '-u', os.path.join(ROOT, 'osad_runner.py'), cmd, json.dumps(args)]
    result, t_start = None, time.time()
    last_progress = t_start - 30.0
    with open(os.path.join(ROOT, 'runner.log'), 'a', encoding='utf-8') as log:
        log.write('\n### %s %s\n' % (cmd, json.dumps(args, ensure_ascii=False)))
        p = subprocess.Popen(cmdline, cwd=os.path.join(ROOT, 'ocp'), env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, errors='replace')
        try:
            for line in p.stdout:
                log.write(line)
                if line.startswith('[osad]'):
                    print(line.rstrip(), flush=True)
                elif line.startswith('RESULT '):
                    result = json.loads(line[len('RESULT '):])
                else:
                    m = IPOPT_LINE.match(line)
                    if m and time.time() - last_progress >= 30.0:
                        last_progress = time.time()
                        print('       IPOPT %4d 回（%3.0f 秒）：目的関数 %s，制約のずれ %s'
                              % (int(m.group(1)), time.time() - t_start, m.group(2), m.group(3)), flush=True)
            p.wait()
        except BaseException:                 # セルの中断など
            p.terminate()
            try:
                p.wait(10)
            except subprocess.TimeoutExpired:
                p.kill()
                p.wait()
            raise
    if p.returncode != 0 or result is None:
        with open(os.path.join(ROOT, 'runner.log'), encoding='utf-8', errors='replace') as f:
            print('\n'.join(f.read().splitlines()[-20:]))
        raise RuntimeError('OpenSimAD の %s が途中で止まりました（ログ：%s/runner.log）' % (cmd, ROOT))
    return result

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

PHASE_STYLE = {'rise': ('tab:green', 'rise'), 'turn': ('tab:purple', 'turn'),
               'turn_back': ('tab:purple', 'turn back'), 'sit': ('tab:brown', 'sit')}


def plot_trial(mot, trc, cands, passes, phases, window=None, title='', window_label='selected step'):
    """股・膝の屈曲角度と踵の高さ（床を水平にしたマーカー）．灰色：直進の区間，色の帯：TUG の区切り，
    縦線：踵接地（実線が右，破線が左），上の数字：cands（start_s・end_s の列を持つ表）の番号，金色：選んだ区間（凡例は window_label）"""
    t = mot['time'].values
    lv = level_trc(trc, floor_plane(trc))
    fig, ax = plt.subplots(3, 1, figsize=(10, 6.4), sharex=True)
    for s, c in (('r', 'tab:red'), ('l', 'tab:blue')):
        ax[0].plot(t, mot['hip_flexion_' + s], c, lw=1.2)
        ax[1].plot(t, mot['knee_angle_' + s], c, lw=1.2)
    ax[2].plot(lv['time'], lv['markers']['r_calc_study'][:, 1], 'tab:red', lw=1.2)
    ax[2].plot(lv['time'], lv['markers']['L_calc_study'][:, 1], 'tab:blue', lw=1.2)
    for a in ax:
        for p in passes:
            a.axvspan(p['t0'], p['t1'], color='0.9', lw=0, zorder=0)
            for x in p['r'][0]:
                a.axvline(x, color='tab:red', lw=0.6, alpha=0.5)
            for x in p['l'][0]:
                a.axvline(x, color='tab:blue', lw=0.6, ls='--', alpha=0.5)
        for name, (x0, x1) in phases.items():
            a.axvspan(x0, x1, color=PHASE_STYLE[name][0], alpha=0.2, lw=0, zorder=0)
        if window is not None:
            a.axvspan(window[0], window[1], color='gold', alpha=0.45, lw=0, zorder=0)
    top = ax[0].get_ylim()[1]
    xlo, xhi = (window[0] - 0.8, window[1] + 0.8) if window is not None else (t[0], t[-1])
    for no, r in cands.iterrows():
        x = (r['start_s'] + r['end_s']) / 2
        if xlo <= x <= xhi:                      # 拡大したときは見える範囲の番号だけ（外の番号で図が広がらないように）
            ax[0].text(x, top, str(no), ha='center', va='top', fontsize=9, fontweight='bold', clip_on=True)
    handles = [Line2D([], [], color='tab:red', label='right'), Line2D([], [], color='tab:blue', label='left'),
               Patch(color='0.9', label='straight pass')]
    seen = set()
    for name in phases:
        c, lab = PHASE_STYLE[name]
        if lab not in seen:
            handles.append(Patch(color=c, alpha=0.2, label=lab))
            seen.add(lab)
    if window is not None:
        handles.append(Patch(color='gold', alpha=0.45, label=window_label))
        ax[0].set_xlim(xlo, xhi)
    ax[0].legend(handles=handles, loc='upper right', fontsize=8, ncol=3, framealpha=0.9)
    ax[0].set_ylabel('Hip flexion (deg)')
    ax[1].set_ylabel('Knee flexion (deg)')
    ax[2].set_ylabel('Heel height (m)')
    ax[2].set_xlabel('Time (s)')
    ax[0].set_title(title)
    fig.tight_layout()
    plt.show()


def plot_result(res, title='', save=None):
    """上：床反力（体重比．鉛直，前後，左右の鉛直の和），中：関節モーメント（Nm/kg），下：関節角度（破線が入力，実線が解）．
    save にパスを渡すと，その画像にも保存する"""
    g, m, k, ik = res['grf'], res['mom'], res['kin'], res['ik']
    t0, t1 = float(g['time'].iloc[0]), float(g['time'].iloc[-1])
    ik = ik[(ik['time'] >= t0 - 0.05) & (ik['time'] <= t1 + 0.05)]
    fig, ax = plt.subplots(3, 3, figsize=(11, 7.4), sharex=True)
    for s, side, c in (('r', 'right', 'tab:red'), ('l', 'left', 'tab:blue')):
        ax[0, 0].plot(g['time'], g['ground_force_%s_vy' % side] / res['bw'], c, label=side)
        ax[0, 1].plot(g['time'], g['ground_force_%s_vx' % side] / res['bw'], c)
        for j, jn in enumerate(JOINTS):
            ax[1, j].plot(m['time'], m['%s_%s_moment' % (jn, s)] / res['mass_kg'], c)
            ax[2, j].plot(ik['time'], ik[jn + '_' + s], c, ls='--', lw=1)
            ax[2, j].plot(k['time'], k[jn + '_' + s], c)
    ax[0, 2].plot(g['time'], (g['ground_force_right_vy'] + g['ground_force_left_vy']) / res['bw'], 'k')
    ax[0, 2].axhline(1.0, color='0.5', lw=0.8, ls=':')
    titles = [['Vertical GRF (BW)', 'Anterior-posterior GRF (BW, + forward)', 'Vertical GRF, R + L (BW)'],
              ['Hip moment (Nm/kg, + flexion)', 'Knee moment (Nm/kg, + flexion)', 'Ankle moment (Nm/kg, + dorsiflexion)'],
              ['Hip flexion (deg)', 'Knee flexion (deg)', 'Ankle dorsiflexion (deg)']]
    for i in range(3):
        for j in range(3):
            ax[i, j].set_title(titles[i][j], fontsize=9)
    ax[0, 0].legend(fontsize=8)
    ax[2, 0].legend(handles=[Line2D([], [], color='k', ls='--', lw=1, label='input (OpenCap)'),
                             Line2D([], [], color='k', label='OpenSimAD')], fontsize=8)
    for a in ax[2]:
        a.set_xlabel('Time (s)')
    ax[0, 0].set_xlim(t0, t1)
    fig.suptitle(title)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=130)
    plt.show()


def plot_activations(kin, window, title='', save=None):
    """右脚の主な筋の筋活動（0〜1）．横軸は区間の %（歩行周期なら右の踵接地が 0 % と 100 %）．save にパスを渡すと保存する．図を返す"""
    t0, t1 = window
    x = (kin['time'].values - t0) / (t1 - t0) * 100
    fig, ax = plt.subplots(2, 4, figsize=(12, 5), sharex=True, sharey=True)
    for a, mname in zip(ax.flat, ACT_MUSCLES):
        a.plot(x, kin[mname + '/activation'].values, 'tab:red')
        a.set_title(ACT_NAMES[mname], fontsize=9)
        a.set_ylim(0, 1)
    for a in ax[1]:
        a.set_xlabel('% of the window')
    for a in ax[:, 0]:
        a.set_ylabel('activation')
    fig.suptitle(title)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=130)
    plt.show()
    return fig




In [ ]:
# OpenSim の環境で使うファイル：osad_runner.py
TOOL_TEXT['osad_runner.py'] = r'''"""Docker 内の OpenSim 環境で前処理（prep），筋駆動の求解（solve），書き出し（export）を実行する．
進捗は [osad]，処理結果は RESULT に続く JSON で出力する．
入力はローカルファイルを使う．
"""
import io
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import yaml

BASE = os.getcwd()
sys.path.insert(0, BASE)
sys.path.insert(0, os.path.join(BASE, 'UtilsDynamicSimulations', 'OpenSimAD'))

TORQUE_WEIGHTS = {'positionTrackingTerm': 10, 'velocityTrackingTerm': 10, 'accelerationTrackingTerm': 50,
                  'armExcitationTerm': 0.001, 'lumbarExcitationTerm': 0.001,
                  'coordinateExcitationTerm': 1, 'jointAccelerationTerm': 0.001}
# opencap-processing の Examples/example_walking_opensimAD.py（72b5416）が追う量にかけるフィルタ（関節角度・速度・加速度を 6 Hz，
# 加速度は速度をスプラインして微分）．"filters": "opencap_example" のときに使う
EXAMPLE_FILTERS = {'filter_Qs_toTrack': True, 'cutoff_freq_Qs': 6, 'filter_Qds_toTrack': True, 'cutoff_freq_Qds': 6,
                   'filter_Qdds_toTrack': True, 'cutoff_freq_Qdds': 6, 'splineQds': True}
STAGES = {}


def say(msg):
    print('[osad] ' + msg, flush=True)


def model_settings(settings, model):
    """駆動のしかた：'torque' はトルク駆動（TORQUE_WEIGHTS），'muscle' は筋駆動（opencap-processing の既定の重みのまま）"""
    if model == 'torque':
        settings['torque_driven_model'] = True
        settings['weights'] = dict(TORQUE_WEIGHTS)
    elif model == 'muscle':
        settings['torque_driven_model'] = False
    else:
        raise ValueError("model は 'torque' か 'muscle'：%r" % (model,))
    return settings


def example_filters(settings):
    """追う量のフィルタを opencap-processing の歩行の例と同じにする"""
    settings.update(EXAMPLE_FILTERS)
    return settings


def write_mot(path, labels, data, title='Coordinates', in_degrees=True):
    """OpenSim の .mot（opencap-processing の numpy_to_storage と同じ見出し）を書く"""
    data = np.asarray(data, dtype=float)
    assert data.shape[1] == len(labels) and labels[0] == 'time'
    bad = [x for x in labels if any(ch.isspace() for ch in x)]
    if bad:
        raise ValueError('列の名前に空白があります（OpenSim は空白で列を分けて数える）：%s' % bad)
    head = [title, 'version=1', 'nRows=%d' % data.shape[0], 'nColumns=%d' % data.shape[1],
            'inDegrees=%s' % ('yes' if in_degrees else 'no'), '',
            'Units are S.I. units (second, meters, Newtons, ...)',
            "If the header above contains a line with 'inDegrees', this indicates whether rotational values are in degrees (yes) "
            "or radians (no).", '', 'endheader']
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(head) + '\n' + '\t'.join(labels) + '\n')
        for row in data:
            f.write('\t'.join('%.8f' % v for v in row) + '\n')


def split_columns(df):
    """OpenSimAD の kinematics_activations の列を，座標と筋活動に分ける．筋活動は OpenSim 4 の状態の名前（/forceset/<筋>/activation）にする"""
    cols = [c for c in df.columns if c != 'time']
    acts = {c: '/forceset/%s/activation' % c[:-len('/activation')] for c in cols if c.endswith('/activation')}
    return [c for c in cols if c not in acts], acts


def external_loads_xml(datafile, sides=(('right', 'calcn_r'), ('left', 'calcn_l'))):
    """足ごとの床反力（OpenSimAD の GRF_resultant：ground_force_<足>_v*，ground_force_<足>_p*，ground_torque_<足>_*）を
    踵骨に加える ExternalLoads の .xml（力と作用点は全体座標）"""
    lines = ['<?xml version="1.0" encoding="UTF-8" ?>', '<OpenSimDocument Version="40000">',
             '  <ExternalLoads name="externalloads">', '    <objects>']
    for side, body in sides:
        lines += ['      <ExternalForce name="%s">' % side,
                  '        <applied_to_body>%s</applied_to_body>' % body,
                  '        <force_expressed_in_body>ground</force_expressed_in_body>',
                  '        <point_expressed_in_body>ground</point_expressed_in_body>',
                  '        <force_identifier>ground_force_%s_v</force_identifier>' % side,
                  '        <point_identifier>ground_force_%s_p</point_identifier>' % side,
                  '        <torque_identifier>ground_torque_%s_</torque_identifier>' % side,
                  '        <data_source_name>Unassigned</data_source_name>',
                  '      </ExternalForce>']
    lines += ['    </objects>', '    <groups />', '    <datafile>%s</datafile>' % datafile, '  </ExternalLoads>',
              '</OpenSimDocument>']
    return '\n'.join(lines) + '\n'


# 逆動力学で計算し直した関節モーメントを OpenSimAD の値と比べる関節と，一致とみなす目安．床反力が二重に入ると，相関は高いまま
# 差が大きくなるため，相関と振れ幅に対する RMS の両方を確認する．
ID_DOFS = ['hip_flexion_r', 'hip_flexion_l', 'knee_angle_r', 'knee_angle_l', 'ankle_angle_r', 'ankle_angle_l']
ID_MIN_CORR, ID_MAX_RMS_REL = 0.95, 0.10


def read_sto_table(path):
    """.mot・.sto の表（endheader の次の行が列の名前．値の前の空白と行の終わりのタブは捨てる）→ (列の名前, 値の配列)"""
    with open(path, encoding='utf-8') as f:
        lines = f.read().splitlines()
    i = next(k for k, line in enumerate(lines) if line.strip().lower() == 'endheader')
    cols = lines[i + 1].split()
    rows = [[float(x) for x in line.split()] for line in lines[i + 2:] if line.strip()]
    return cols, np.array(rows, dtype=float).reshape(-1, len(cols))


def moment_agreement(id_cols, id_data, ref_cols, ref_data, dofs=None):
    """逆動力学で計算し直した関節モーメント（id）と，OpenSimAD の関節モーメント（ref）の一致．id を ref の時刻へ線形補間し
    （重なる時刻だけ），関節ごとに相関と，差の RMS を ref の振れ幅で割った値を求める．
    ok：比べる関節がすべてあり，相関の最小が ID_MIN_CORR 以上で，差の RMS の比の最大が ID_MAX_RMS_REL 以下"""
    dofs = ID_DOFS if dofs is None else dofs
    t_id, t_ref = id_data[:, 0], ref_data[:, 0]
    keep = (t_ref >= t_id.min() - 1e-9) & (t_ref <= t_id.max() + 1e-9)
    res = {}
    for d in dofs:
        c = d + '_moment'
        if c not in id_cols or c not in ref_cols:
            continue
        ref = ref_data[keep, ref_cols.index(c)]
        est = np.interp(t_ref[keep], t_id, id_data[:, id_cols.index(c)])
        rng = float(np.ptp(ref))
        corr = float(np.corrcoef(est, ref)[0, 1]) if np.std(est) > 0 and np.std(ref) > 0 else float('nan')
        rms = float(np.sqrt(np.mean((est - ref) ** 2)))
        res[d] = {'corr': corr, 'rms_rel': rms / rng if rng > 0 else float('inf')}
    min_corr = min((v['corr'] for v in res.values()), default=float('nan'))
    max_rel = max((v['rms_rel'] for v in res.values()), default=float('nan'))
    ok = bool(len(res) == len(dofs) and np.isfinite(min_corr) and min_corr >= ID_MIN_CORR and max_rel <= ID_MAX_RMS_REL)
    return {'ok': ok, 'min_corr': min_corr, 'max_rms_rel': max_rel, 'dofs': res, 'rows': int(keep.sum())}


def read_mot_lines(path):
    """.mot の見出し（endheader まで）と表"""
    with open(path, encoding='latin-1') as f:
        lines = f.read().splitlines()
    k = next(i for i, x in enumerate(lines) if x.strip().lower() == 'endheader')
    df = pd.read_csv(io.StringIO('\n'.join(lines[k + 1:])), sep='\t')
    # 行の最後のタブ（opencap-processing の numpy_to_storage が付ける）でできる空の列を除く
    empty = [c for c in df.columns if str(c).startswith('Unnamed') and df[c].isna().all()]
    return lines[:k + 1], df.drop(columns=empty)


def rx(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def ry(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def rz(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def decompose(M):
    """M = Rz(tilt) Rx(list) Ry(rotation) の角度（ground_pelvis の回転の順．OpenSim で確かめた）"""
    return np.arctan2(-M[0, 1], M[1, 1]), np.arcsin(np.clip(M[2, 1], -1, 1)), np.arctan2(-M[2, 0], M[2, 2])


class Pose:
    """OpenSim のモデルに .mot の1行の座標を入れて，マーカーや接触球の位置を求める"""

    def __init__(self, model_path):
        import opensim as osim
        osim.Logger.setLevelString('error')
        self.osim = osim
        self.model = osim.Model(model_path)
        self.state = self.model.initSystem()
        cs = self.model.getCoordinateSet()
        self.coords = [cs.get(i) for i in range(cs.getSize())]
        self.rot = {c.getName() for c in self.coords if c.getMotionType() == 1}

    def set(self, row):
        for c in self.coords:
            n = c.getName()
            if n in row:
                v = float(row[n])
                c.setValue(self.state, np.radians(v) if n in self.rot else v, False)
        self.model.realizePosition(self.state)


def model_path(session, trial, name):
    for p in (os.path.join(session, 'OpenSimData', 'Model', trial, name), os.path.join(session, 'OpenSimData', 'Model', name)):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(name)


def level_floor(session, trial, model, lib):
    """体全体（骨盤の位置と向き）を原点のまわりに回して床を水平にし，<試行>_level.mot を書く．関節角度は変えない．
    床の面は，モデルの踵のマーカー（r_calc_study，L_calc_study．順運動学で求める）から floor_plane で求める．
    戻り値：前後の傾きなど．床の面を求められないとき（その場の動作）は None"""
    sys.path.insert(0, lib)
    import oc_research_lite as R
    kin = os.path.join(session, 'OpenSimData', 'Kinematics')
    head, df = read_mot_lines(os.path.join(kin, trial + '.mot'))
    P = Pose(model_path(session, trial, model + '_scaled.osim'))
    mk = {n: P.model.getMarkerSet().get(n) for n in ('r_calc_study', 'L_calc_study')}

    def heels(frame):
        out = {n: np.empty((len(frame), 3)) for n in mk}
        for i, (_, row) in enumerate(frame.iterrows()):
            P.set(row)
            for n, m in mk.items():
                v = m.getLocationInGround(P.state)
                out[n][i] = [v.get(0), v.get(1), v.get(2)]
        return out

    fs = 1.0 / np.median(np.diff(df['time'].values))
    plane = R.floor_plane({'time': df['time'].values, 'rate': fs, 'markers': heels(df)})
    if not np.isfinite(plane['tilt_deg']):
        return None
    L = R.level_rotation(plane)
    ang = np.radians(df[['pelvis_tilt', 'pelvis_list', 'pelvis_rotation']].values)
    new = np.array([decompose(L @ rz(a) @ rx(b) @ ry(c)) for a, b, c in ang])
    new = ang + (new - ang + np.pi) % (2 * np.pi) - np.pi          # 元の角度と連続するように
    out = df.copy()
    out[['pelvis_tilt', 'pelvis_list', 'pelvis_rotation']] = np.degrees(new)
    out[['pelvis_tx', 'pelvis_ty', 'pelvis_tz']] = df[['pelvis_tx', 'pelvis_ty', 'pelvis_tz']].values @ L.T
    with open(os.path.join(kin, trial + '_level.mot'), 'w', encoding='latin-1') as f:
        f.write('\n'.join(head) + '\n')
        out.to_csv(f, sep='\t', index=False, float_format='%.8f', lineterminator='\n')
    after = R.floor_plane({'time': out['time'].values, 'rate': fs, 'markers': heels(out)})
    mdir = os.path.join(session, 'OpenSimData', 'Model')            # 1台版：_level の試行も同じモデルを見る
    if os.path.isdir(os.path.join(mdir, trial)) and not os.path.exists(os.path.join(mdir, trial + '_level')):
        os.symlink(trial, os.path.join(mdir, trial + '_level'))
    return {'tilt_before_deg': float(plane['tilt_deg']), 'tilt_after_deg': float(after['tilt_deg']),
            'pelvis_angle_change_deg': float(np.degrees(np.abs(new - ang)).max())}


def foot_low(session, trial, model, t0, t1):
    """接触球のついたモデルで，区間の各コマの左右の足の最も低い点（球の中心の高さ − 半径，m）の，下から 10% の値"""
    _, df = read_mot_lines(os.path.join(session, 'OpenSimData', 'Kinematics', trial + '.mot'))
    df = df[(df['time'] >= t0 - 1e-6) & (df['time'] <= t1 + 1e-6)]
    P = Pose(model_path(session, trial, model + '_scaled_adjusted_contacts.osim'))
    gs = P.model.getContactGeometrySet()
    sph = []
    for i in range(gs.getSize()):
        c = P.osim.ContactSphere.safeDownCast(gs.get(i))
        if c:
            sph.append(('r' if gs.get(i).getName().endswith('_r') else 'l', c.getFrame(), c.get_location(), c.getRadius()))
    low = {'r': [], 'l': []}
    for _, row in df.iterrows():
        P.set(row)
        h = {'r': np.inf, 'l': np.inf}
        for s, fr, loc, rad in sph:
            h[s] = min(h[s], fr.findStationLocationInGround(P.state, loc).get(1) - rad)
        for s in h:
            low[s].append(h[s])
    q = {s: float(np.percentile(v, 10)) for s, v in low.items()}
    return {'r': q['r'], 'l': q['l'], 'diff': q['r'] - q['l']}


def clear_case(session, trial, case):
    """同じ case の古い結果（前に解いたときのファイル）を消す．解き直して収束しなかったとき，古い結果を読まないように"""
    import glob
    for p in glob.glob(os.path.join(session, 'OpenSimData', 'Dynamics', trial, '*_%s.*' % case)):
        os.remove(p)


def patch_utils(U, timers):
    def no_download(*a, **k):
        raise RuntimeError('.mot が見つかりません（このノートブックは OpenCap のサーバーに接続しません）')
    U.download_kinematics = no_download
    if not timers:
        return

    def timed(name, label, f):
        def g(*x, **k):
            s = time.time()
            say(label + 'を始めます')
            r = f(*x, **k)
            STAGES[name] = round(time.time() - s, 1)
            say('%s：%.0f 秒' % (label, STAGES[name]))
            return r
        return g
    for name, label in (('adjust_muscle_wrapping', '筋の走行の調整'), ('generate_model_with_contacts', '足の接触球'),
                        ('generateExternalFunction', '外部関数（OpenSimAD のライブラリでコンパイル）')):
        setattr(U, name, timed(name, label, getattr(U, name)))


def session_model(session):
    with open(os.path.join(session, 'sessionMetadata.yaml'), encoding='utf-8') as f:
        return (yaml.safe_load(f) or {}).get('openSimModel', 'LaiUhlrich2022')


def prep(a):
    import utilsOpenSimAD as U
    session = os.path.join(a['data'], a['sid'])
    out = {'trial': a['trial'], 'level': None, 'stages': STAGES}
    if a.get('level', True):
        say('床を水平にします（モデルの踵のマーカーから床の面を求め，体全体を回す）')
        lv = level_floor(session, a['trial'], session_model(session), a['lib'])
        if lv is None:
            say('床の面を求められませんでした（その場の動作）．水平にしません')
        else:
            out.update(trial=a['trial'] + '_level', level=lv)
            say('床の傾き %.2f° → %.2f°' % (lv['tilt_before_deg'], lv['tilt_after_deg']))
    patch_utils(U, timers=True)
    U.processInputsOpenSimAD(BASE, a['data'], a['sid'], out['trial'], 'walking', [-1, -1],
                             treadmill_speed=0, contact_side='all')
    print('RESULT ' + json.dumps(out), flush=True)


def solve(a):
    import casadi
    import utilsOpenSimAD as U
    patch_utils(U, timers=False)
    import mainOpenSimAD as M
    session = os.path.join(a['data'], a['sid'])
    clear_case(session, a['trial'], a['case'])
    settings = U.processInputsOpenSimAD(BASE, a['data'], a['sid'], a['trial'], 'walking', [a['t0'], a['t1']],
                                        treadmill_speed=0, contact_side='all')
    foot = foot_low(session, a['trial'], settings['OpenSimModel'], a['t0'], a['t1'])
    say('足の最も低い点（区間の下から 10%%）：右 %.3f m，左 %.3f m，差 %.1f cm' % (foot['r'], foot['l'], 100 * foot['diff']))
    settings['meshDensity'] = int(a['mesh'])
    model_settings(settings, a.get('model', 'torque'))
    if a.get('filters') == 'opencap_example':
        example_filters(settings)
    nlpsol, limit = casadi.nlpsol, 60.0 * float(a['max_min'])

    def nlpsol_with_limit(name, solver, prob, opts):
        opts = dict(opts)
        opts['ipopt.max_wall_time'] = limit
        return nlpsol(name, solver, prob, opts)
    U.ca.nlpsol = nlpsol_with_limit
    say('求解を始めます：%.3f〜%.3f 秒，%s，メッシュ %d，打ち切り %g 分%s'
        % (a['t0'], a['t1'], '筋駆動' if a.get('model') == 'muscle' else 'トルク駆動', a['mesh'], a['max_min'],
           '，追う量のフィルタは opencap-processing の歩行の例と同じ' if a.get('filters') == 'opencap_example' else ''))
    s = time.time()
    M.run_tracking(BASE, a['data'], a['sid'], settings, case=a['case'], solveProblem=True, analyzeResults=True,
                   computeKAM=False, computeMCF=False)
    out = {'case': a['case'], 'solve_s': round(time.time() - s, 1), 'foot_low_m': foot,
           'iter_count': None, 'return_status': 'unknown', 'success': False}
    sp = os.path.join(session, 'OpenSimData', 'Dynamics', a['trial'], 'stats_%s.npy' % a['case'])
    if os.path.exists(sp):
        st = np.load(sp, allow_pickle=True).item()      # いまの求解が書いたファイル
        out.update(iter_count=int(st.get('iter_count', -1)), return_status=str(st.get('return_status')),
                   success=bool(st.get('success')))
    print('RESULT ' + json.dumps(out), flush=True)


def export(a):
    """結果を OpenSim形式で a['out'] へ書き出し，OpenSim の API で読み込めるかを確かめる．
    motion.mot（座標だけ），motion_activations.mot（座標と筋活動．筋活動は /forceset/<筋>/activation），GRF.mot と
    GRF_ExternalLoads.xml（足ごとの床反力を踵骨に），kinetics.mot（関節モーメント），モデル（筋の走行を調整し足の接触球をつけたもの．
    接触球の力を切った写し），ID_setup.xml（筋を除く逆動力学の設定），opensimad_original/（OpenSimAD が書いたファイルとモデルそのまま）．
    逆動力学を設定ファイルから計算し直し，OpenSimAD の関節モーメントと比べる（moment_agreement）"""
    import shutil
    import opensim
    session = os.path.join(a['data'], a['sid'])
    trial, case, out = a['trial'], a['case'], a['out']
    d = os.path.join(session, 'OpenSimData', 'Dynamics', trial)
    orig = os.path.join(out, 'opensimad_original')
    os.makedirs(orig, exist_ok=True)
    src = {k: os.path.join(d, '%s_%s_%s.mot' % (k, trial, case)) for k in ('kinematics_activations', 'GRF_resultant', 'GRF', 'kinetics')}
    for p in list(src.values()) + [os.path.join(d, 'stats_%s.npy' % case)]:
        if os.path.exists(p):
            shutil.copy(p, orig)
    name = lambda kind: os.path.join(out, '%s_%s_%s' % (trial, case, kind))  # noqa: E731
    _, kin = read_mot_lines(src['kinematics_activations'])
    coords, acts = split_columns(kin)
    files = {'motion': name('motion.mot'), 'motion_activations': name('motion_activations.mot'), 'GRF': name('GRF.mot'),
             'external_loads': name('GRF_ExternalLoads.xml'), 'kinetics': name('kinetics.mot')}
    write_mot(files['motion'], ['time'] + coords, kin[['time'] + coords].values)
    write_mot(files['motion_activations'], ['time'] + coords + list(acts.values()), kin[['time'] + coords + list(acts)].values)
    shutil.copy(src['GRF_resultant'], files['GRF'])
    shutil.copy(src['kinetics'], files['kinetics'])
    with open(files['external_loads'], 'w', encoding='utf-8') as f:
        f.write(external_loads_xml(os.path.basename(files['GRF'])))
    model = session_model(session) + '_scaled_adjusted_contacts.osim'
    files['model_original'] = os.path.join(orig, model)
    shutil.copy(os.path.join(session, 'OpenSimData', 'Model', model), files['model_original'])
    # 表示と逆動力学などのツールに使うモデル：接触球の力を切る（力が有効なまま ExternalLoads を足すと，床反力が二重に入る）．球は表示される
    files['model'] = os.path.join(out, session_model(session) + '_scaled_adjusted_contacts_disabled.osim')
    mo = opensim.Model(files['model_original'])
    fo = mo.updForceSet()
    n_contacts = 0
    for i in range(fo.getSize()):
        fc = fo.get(i)
        if fc.getConcreteClassName() == 'SmoothSphereHalfSpaceForce':
            fc.set_appliesForce(False)
            n_contacts += 1
    mo.printToXML(files['model'])
    # OpenSim の API で読み込めるか
    m = opensim.Model(files['model'])
    m.initSystem()
    fs = m.getForceSet()
    contact_on = [fs.get(i).get_appliesForce() for i in range(fs.getSize())
                  if fs.get(i).getConcreteClassName() == 'SmoothSphereHalfSpaceForce']
    cs = m.getCoordinateSet()
    model_coords = {cs.get(i).getName() for i in range(cs.getSize())}
    sv = m.getStateVariableNames()
    states = {sv.get(i) for i in range(sv.getSize())}
    # Storage は変数に入れておく（一時的なオブジェクトから列の名前の参照を取ると，すぐに消えて落ちる）
    stores = {k: opensim.Storage(files[k]) for k in ('motion', 'motion_activations', 'GRF', 'kinetics')}
    rows = {k: int(st.getSize()) for k, st in stores.items()}
    labels = stores['GRF'].getColumnLabels()
    grf_cols = {labels.get(i) for i in range(labels.getSize())}
    el = opensim.ExternalLoads(files['external_loads'], True)
    loads = []
    for i in range(el.getSize()):
        ef = el.get(i)
        need = ([ef.get_force_identifier() + x for x in 'xyz'] + [ef.get_point_identifier() + x for x in 'xyz']
                + [ef.get_torque_identifier() + x for x in 'xyz'])
        loads.append({'name': ef.getName(), 'body': ef.get_applied_to_body(),
                      'body_in_model': bool(m.getBodySet().contains(ef.get_applied_to_body())),
                      'columns_in_file': all(c in grf_cols for c in need)})
    # 逆動力学の設定ファイル（このフォルダの中の名前で書く．筋を除く）を書き，それを読み込んで計算し直し，OpenSimAD の関節モーメントと比べる
    files['id_setup'] = name('ID_setup.xml')
    id_out = '%s_%s_ID.sto' % (trial, case)
    tool = opensim.InverseDynamicsTool()
    tool.setName('%s_%s' % (trial, case))
    tool.setModelFileName(os.path.basename(files['model']))
    tool.setCoordinatesFileName(os.path.basename(files['motion']))
    tool.setExternalLoadsFileName(os.path.basename(files['external_loads']))
    tool.setStartTime(max(float(kin['time'].iloc[0]), stores['GRF'].getFirstTime()))
    tool.setEndTime(min(float(kin['time'].iloc[-1]), stores['GRF'].getLastTime()))
    excluded = opensim.ArrayStr()
    excluded.append('Muscles')
    tool.setExcludedForces(excluded)
    tool.setLowpassCutoffFrequency(-1)
    tool.setResultsDir('.')
    tool.setOutputGenForceFileName(id_out)
    tool.printToXML(files['id_setup'])
    cwd = os.getcwd()
    os.chdir(out)                                   # 設定ファイルの中の名前は，このフォルダからの相対
    try:
        id_tool = opensim.InverseDynamicsTool(os.path.basename(files['id_setup']))
        id_ok = bool(id_tool.run())
    finally:
        os.chdir(cwd)
    agree = None
    if id_ok and os.path.exists(os.path.join(out, id_out)):
        ic, idd = read_sto_table(os.path.join(out, id_out))
        kc, kd = read_sto_table(files['kinetics'])
        agree = moment_agreement(ic, idd, kc, kd)
    if os.path.exists(os.path.join(out, id_out)):
        os.remove(os.path.join(out, id_out))        # 確かめるためだけの出力は残さない（設定ファイルで作り直せる）
    nan = float('nan')
    checks = {'coordinates_in_model': all(c in model_coords for c in coords), 'n_coordinates': len(coords),
              'activations_in_model': all(v in states for v in acts.values()), 'n_muscles': len(acts),
              'rows': rows, 'loads': loads, 'time': [float(kin['time'].iloc[0]), float(kin['time'].iloc[-1])],
              'contacts_disabled': bool(contact_on) and not any(contact_on), 'n_contacts': n_contacts}
    checks['id_tool'] = {'ok': id_ok, 'excluded': ['Muscles'], 'agree': bool(agree and agree['ok']),
                         'rows': agree['rows'] if agree else 0, 'min_corr': agree['min_corr'] if agree else nan,
                         'max_rms_rel': agree['max_rms_rel'] if agree else nan, 'dofs': agree['dofs'] if agree else {}}
    ok = (checks['coordinates_in_model'] and checks['activations_in_model'] and all(rows.values())
          and all(x['body_in_model'] and x['columns_in_file'] for x in loads) and checks['contacts_disabled']
          and id_ok and checks['id_tool']['agree'])
    print('RESULT ' + json.dumps({'files': {k: os.path.relpath(v, out) for k, v in files.items()}, 'checks': checks,
                                  'ok': bool(ok)}), flush=True)


if __name__ == '__main__':
    {'prep': prep, 'solve': solve, 'export': export}[sys.argv[1]](json.loads(sys.argv[2]))
'''

In [ ]:
# OpenSim の環境で使うファイル：oc_research_lite.py
TOOL_TEXT['oc_research_lite.py'] = r'''"""osad_runner.py が使う床面の推定と水平化．"""

import numpy as np
from scipy import signal


def lowpass(x, fs, fc=6.0, order=4):
    """ゼロ位相のバターワース低域通過フィルタ（列ごと）．OpenCap の立ち上がりの解析は 6 Hz を使う．"""
    b, a = signal.butter(order, fc / (fs / 2))
    return signal.filtfilt(b, a, np.asarray(x, dtype=float), axis=0)


UP = np.array([0.0, 1.0, 0.0])


def floor_plane(trc, vthr=0.15, min_spread=1.0):
    """床面 y = a x + b z + c を，接地中（速さ vthr m/s 未満）の左右の踵マーカーから最小二乗で求める．
    キャリブレーションのチェッカーボードが傾いていたり小さく写っていたりすると，床が傾いて再構成される．
    踵が min_spread m 以上動いた向きの傾きだけを求める（その場の動作では傾きを求められず nan）．
    戻り値：a，b，c（c は踵マーカーの高さなので床より数 cm 上），傾き tilt_deg，平面からのずれ rms_m"""
    fs, P = trc['rate'], []
    for k in ('r_calc_study', 'L_calc_study'):
        x = lowpass(trc['markers'][k], fs)
        v = np.linalg.norm(np.gradient(x, axis=0) * fs, axis=1)
        P.append(x[v < vthr])
    P = np.vstack(P)
    H = P[:, [0, 2]] - P[:, [0, 2]].mean(axis=0)
    U = np.linalg.svd(H, full_matrices=False)[2]        # 踵が動いた主な向き（水平面内）
    S = H @ U.T
    use = [k for k in range(2) if np.subtract(*np.percentile(S[:, k], [95, 5])) >= min_spread]
    if not use:
        return {'a': np.nan, 'b': np.nan, 'c': np.median(P[:, 1]), 'tilt_deg': np.nan,
                'rms_m': np.std(P[:, 1]), 'n': len(P)}
    A = np.column_stack([S[:, use], np.ones(len(P))])
    coef = np.linalg.lstsq(A, P[:, 1], rcond=None)[0]
    g = np.zeros(2)
    g[use] = coef[:-1]
    a, b = g @ U                                         # 傾きを x，z の向きに戻す
    c = coef[-1] - a * P[:, 0].mean() - b * P[:, 2].mean()
    rms = np.sqrt(np.mean((P[:, 1] - (a * P[:, 0] + b * P[:, 2] + c)) ** 2))
    return {'a': a, 'b': b, 'c': c, 'tilt_deg': np.degrees(np.arctan(np.hypot(a, b))), 'rms_m': rms, 'n': len(P)}


def level_rotation(plane):
    """床面の法線を鉛直（Y）に重ねる回転行列．傾きを求められなかったときは回さない"""
    if not np.isfinite(plane['a']):
        return np.eye(3)
    n = np.array([-plane['a'], 1.0, -plane['b']])
    n = n / np.linalg.norm(n)
    v, c = np.cross(n, UP), n @ UP
    s = np.linalg.norm(v)
    if s < 1e-12:
        return np.eye(3)
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx * ((1 - c) / s ** 2)
'''


In [ ]:
# 2. 準備：OpenSim の環境を確かめ，OpenSimAD を動かすファイルを置く
if not os.path.exists(os.path.join(ROOT, 'READY')):
    raise RuntimeError('OpenSim の環境（%s）が見つかりません．Docker 内の /opt/osad と OSAD_ROOT を確認してください' % ROOT)
write_tools(TOOL_TEXT)
ver = subprocess.run([os.path.join(ROOT, 'env', 'bin', 'python'), '-c', 'import opensim, casadi; print(opensim.GetVersion(), casadi.__version__)'],
                     capture_output=True, text=True)
print('OpenSim の環境：%s（OpenSim と CasADi の版：%s）' % (ROOT, ver.stdout.strip() or ver.stderr.strip()[-300:]))
for d in ('figures', 'results'):
    shutil.rmtree(os.path.join(ROOT, d), ignore_errors=True)       # 前に解いたときの図と書き出しを消す
TIMES, T_START = {}, time.time()

## 3. データを読む

zip なら，使うファイルだけを作業のフォルダに展開します．試行の一覧と，使う体重・身長を表示します．

In [ ]:
# 3. データを読む
SID = 'mydata'
reset_session(SID)                                                  # 前のデータの前処理の結果を消す
if os.path.isdir(SESSION):
    SRC = SESSION
elif os.path.isfile(SESSION) and zipfile.is_zipfile(SESSION):
    SRC = os.path.join(ROOT, 'input', SID)
    print('zip から %d のファイルを展開しました（動画は展開しません）' % extract_session(SESSION, SRC))
else:
    raise FileNotFoundError('SESSION が見つかりません：%s（コンテナの中のパスで書きます．データのフォルダは /data）' % SESSION)
INFO = session_info(SRC)
display(INFO['trials'])
if TRIAL not in set(INFO['trials']['trial']):
    raise ValueError('1 節の TRIAL に，上の表の試行の名前を書いてください（いま：%r）' % TRIAL)
if not INFO['trials'].set_index('trial').loc[TRIAL, 'mot']:
    raise ValueError('%s には関節角度（OpenSimData/Kinematics/%s.mot）がありません' % (TRIAL, TRIAL))
MASS = float(MASS_KG) if MASS_KG else INFO['mass_kg']
HEIGHT = float(HEIGHT_M) if HEIGHT_M else INFO['height_m']
check_body(MASS, HEIGHT, where='1 節の入力のセル')
print('体重 %.1f kg，身長 %.2f m，モデル %s' % (float(MASS), float(HEIGHT), INFO['model']))


## 4. 区間を選ぶ

直進の区間の中の歩行周期を表にし，1 節の `CYCLE`・`WINDOW` に従って選びます（決め方は 1 節）．
表の「自動で選べる」が × の周期も，`CYCLE` に番号を書けば解けます．
図の金色が選んだ区間，縦線が踵接地（実線が右，破線が左），灰色が直進の区間，上の数字が表の番号です．

In [ ]:
# 4. 区間を選ぶ
MOT = read_mot(os.path.join(SRC, 'OpenSimData', 'Kinematics', TRIAL + '.mot'))
TRC = read_trc(os.path.join(SRC, 'MarkerData', TRIAL + '.trc'))
PASSES, PHASES = gait_analysis(TRC, MOT)
RISE_END = PHASES.get('rise', (0.0, 0.0))[1]
ONSET = turn_onset(TRC, start=RISE_END)
IS_TUG = 'rise' in PHASES and np.isfinite(ONSET)
CYCLES = walkout_cycles(PASSES, PHASES, ONSET) if IS_TUG else steady_cycles(PASSES, PHASES)
if IS_TUG:
    print('TUG として扱います（立ち上がりの終わり %.2f 秒，方向転換の始まり %.2f 秒）' % (RISE_END, ONSET))
display(pd.DataFrame({'区間': CYCLES['pass'], '足': CYCLES['side'].map({'R': '右', 'L': '左'}), '開始（秒）': CYCLES['start_s'].round(3),
                      '終了（秒）': CYCLES['end_s'].round(3), '長さ（秒）': CYCLES['duration_s'].round(3),
                      '自動で選べる': CYCLES['ok'].map({True: '○', False: '×'}), '理由': CYCLES['note']}, index=CYCLES.index))
if WINDOW:
    chk = check_window(float(WINDOW[0]), float(WINDOW[1]), MOT['time'].values, PASSES, PHASES)
    if chk['errors']:
        raise ValueError('1 節の WINDOW：' + '／'.join(chk['errors']))
    for n in chk['notes']:
        print('注意：' + n)
    T0, T1, SIDE, RULE = chk['t0'], chk['t1'], None, '1 節の WINDOW で指定'
elif CYCLE != 'auto':
    if int(CYCLE) not in CYCLES.index:
        raise ValueError('1 節の CYCLE：表に %d 番はありません' % int(CYCLE))
    r = CYCLES.loc[int(CYCLE)]
    T0, T1, SIDE, RULE = float(r['start_s']), float(r['end_s']), r['side'], '表の %d 番（1 節の CYCLE）' % int(CYCLE)
    if not r['ok']:
        print('注意：自動では選ばない周期です（%s）' % r['note'])
else:
    st = choose_cycle(CYCLES) if IS_TUG else choose_steady(CYCLES, PASSES)
    T0, T1, SIDE = st['t0'], st['t1'], st['side']
    RULE = '自動：TUG の往路の規則' if IS_TUG else '自動：ふつうの歩行の規則'
WHAT = ('%sの歩行周期' % {'R': '右', 'L': '左'}[SIDE]) if SIDE else '区間'
print('使う区間：%s %.3f〜%.3f 秒（%.3f 秒．%s）' % (WHAT, T0, T1, T1 - T0, RULE))
plot_trial(MOT, TRC, CYCLES, PASSES, PHASES, window=(T0, T1), title='%s: %.3f-%.3f s' % (TRIAL, T0, T1),
           window_label='selected gait cycle' if SIDE else 'selected window')

## 5. 筋駆動の OpenSimAD で解く

`LEVEL_FLOOR = True` の場合は床を水平にしてから，選んだ区間を筋駆動で解きます．
IPOPT の進捗は約30秒間隔で表示し，詳細は `runner.log` に保存します．
設定は opencap-processing の公式の歩行の例（`Examples/example_walking_opensimAD.py`）にそろえます：
OpenSimAD が追う入力の関節角度・速度・加速度に 6 Hz のフィルタをかけ，加速度は速度をスプラインして微分して求めます．
例は周期の制約（区間の始めと終わりの状態を同じにする）も入れていますが，ここでは入れません（周期的でない歩行も解けるように）．
同じ設定でも，実行する環境によって IPOPT の反復の道すじが変わり，結果が少し変わります．

In [ ]:
# 5. 筋駆動の OpenSimAD で解く
SESSION_DIR = prepare_session(SRC, SID, TRIAL, MASS, HEIGHT, INFO['model'])
t = time.time()
PREP = run_osad('prep', {'data': os.path.join(ROOT, 'Data'), 'sid': SID, 'trial': TRIAL, 'level': bool(LEVEL_FLOOR), 'lib': ROOT})
TIMES['前処理（床の水平化，筋の走行の調整，接触球，外部関数）'] = time.time() - t
OS_TRIAL = PREP['trial']
CASE = ('walk_%s_m%d' % (SIDE.lower(), int(MESH))) if SIDE else ('window_m%d' % int(MESH))
t = time.time()
OUT = run_osad('solve', {'data': os.path.join(ROOT, 'Data'), 'sid': SID, 'trial': OS_TRIAL, 't0': T0, 't1': T1, 'mesh': int(MESH),
                         'case': CASE, 'max_min': float(TIME_LIMIT_MIN), 'model': 'muscle', 'filters': 'opencap_example'})
TIMES['求解（筋の走行の近似式を作る時間を含む）'] = time.time() - t
print('結果：%s，%s 回，%.1f 分' % (OUT['return_status'], OUT['iter_count'], TIMES['求解（筋の走行の近似式を作る時間を含む）'] / 60))

## 6. 結果を見る

- 1 枚目：床反力（体重比），関節モーメント（体重 1 kg あたり），関節角度（破線が入力の OpenCap，実線が OpenSimAD の解）．
- 2 枚目：右脚の主な筋の筋活動（0〜1）．横軸は区間の %．

**確認**：収束しても，歩行としてありえない値なら，その結果は解釈には使えません．

| 確認 | 目安 |
|---|---|
| 収束 | IPOPT が `Solve_Succeeded` で終わる |
| 鉛直床反力のピーク | 体重の 2 倍以下 |
| 区間の鉛直床反力の平均（左右の和） | 体重の 0.8〜1.2 倍 |
| 入力の角度からのずれ（6関節の RMSE の最大） | 8° 未満 |

**読み方の注意**：区間の端（始まりと終わり）は解釈に使わないでください．メッシュが粗いと，曲線はかくかくして見えます．
筋活動が短い時間だけ 1 近くまで跳ねることがあります．筋活動の大きさは，そのまま解釈しないでください．

In [ ]:
# 6. 結果を見る
RES = (read_result(SESSION_DIR, OS_TRIAL, CASE, MASS, os.path.join(SESSION_DIR, 'OpenSimData', 'Kinematics', OS_TRIAL + '.mot'))
       if OUT['return_status'] == 'Solve_Succeeded' else None)
OK, REASONS = verdict(OUT['return_status'], RES)
print('確認：' + ('通りました' if OK else '当たりました → ' + '／'.join(REASONS)))
if RES:
    print('鉛直床反力のピーク：右 %.2f，左 %.2f 体重比．区間の平均（左右の和）：%.2f 体重比．入力の角度からのずれの最大：%.1f°（%s）'
          % (RES['peak_r'], RES['peak_l'], RES['mean_bw'], RES['dev_max'], RES['dev_joint']))
    TITLE = '%s %s %.3f-%.3f s, muscle-driven, mesh %d' % (TRIAL, ('%s gait cycle' % SIDE) if SIDE else 'window', T0, T1, int(MESH))
    FIGDIR = os.path.join(ROOT, 'figures')
    os.makedirs(FIGDIR, exist_ok=True)
    plot_result(RES, TITLE, save=os.path.join(FIGDIR, 'grf_moments_angles.png'))
    _ = plot_activations(RES['kin'], (T0, T1), TITLE, save=os.path.join(FIGDIR, 'muscle_activations.png'))

## 7. OpenSim形式で書き出す

関節角度，筋活動，床反力，関節モーメント，モデル，逆動力学の設定を保存します．
出力ファイルの説明は結果フォルダの `README.md` に記録します．

OpenSim の API で座標・筋・床反力の整合性を確認します．また，逆動力学を再計算し，
股・膝・足関節の左右で相関 0.95 以上，差の RMS が振れ幅の 10 % 以下か確認します．
床反力の二重計上を避けるため，書き出すモデルでは接触球の力を無効にし，逆動力学では筋を除きます．


In [ ]:
# 7. OpenSim で読み込める形に書き出す
NAME = '%s_%s' % (TRIAL, CASE)
RESDIR = os.path.join(ROOT, 'results', NAME)
shutil.rmtree(RESDIR, ignore_errors=True)
t = time.time()
EXP, WHY = None, ''
if OUT['return_status'] == 'Solve_Succeeded':
    try:
        EXP = run_osad('export', {'data': os.path.join(ROOT, 'Data'), 'sid': SID, 'trial': OS_TRIAL, 'case': CASE, 'out': RESDIR})
    except RuntimeError as e:
        WHY = '書き出しに失敗しました（%s）' % e
else:
    WHY = '求解が収束しなかった（%s）ので，OpenSim の形には書き出しません（OpenSimAD は結果の .mot を書きません）' % OUT['return_status']
if EXP is None:
    print(WHY + '．OpenSimAD が書いた結果のファイル（.mot，.npy）だけを保存します')
    os.makedirs(os.path.join(RESDIR, 'opensimad_original'), exist_ok=True)
    for pat in ('*.mot', '*.npy'):                                 # 設定の .yaml（身長・体重を含む）は入れない
        for p in glob.glob(os.path.join(SESSION_DIR, 'OpenSimData', 'Dynamics', OS_TRIAL, pat)):
            if CASE in os.path.basename(p):
                shutil.copy(p, os.path.join(RESDIR, 'opensimad_original'))
TIMES['書き出し'] = time.time() - t
if EXP:
    chk = EXP['checks']
    print('書き出し確認：%s（座標 %d，筋 %d）' %
          ('通過' if EXP['ok'] else '要確認', chk['n_coordinates'], chk['n_muscles']))
    print('逆動力学：%s，相関の最小 %.3f，差の RMS の最大 %.1f %%' %
          ('一致' if chk['id_tool']['agree'] else '不一致',
           chk['id_tool']['min_corr'], 100 * chk['id_tool']['max_rms_rel']))
    if not EXP['ok']:
        display(chk)
TIMES['全体（2 節から書き出しまで）'] = time.time() - T_START
FIGS = []
if os.path.isdir(os.path.join(ROOT, 'figures')):
    shutil.copytree(os.path.join(ROOT, 'figures'), os.path.join(RESDIR, 'figures'), dirs_exist_ok=True)
    FIGS = ['figures/' + x for x in sorted(os.listdir(os.path.join(RESDIR, 'figures')))]
lv = PREP.get('level')
FLOOR = ('床を水平にしてから解いた（傾き %.2f° → %.2f°．試行の名前は %s）' % (lv['tilt_before_deg'], lv['tilt_after_deg'], OS_TRIAL)
         if lv else '床は水平にしていない')
nan = float('nan')
INFO_RES = {'trial': TRIAL, 'side': SIDE, 't0': T0, 't1': T1, 'mesh': int(MESH), 'model': 'muscle',
            'status': OUT['return_status'], 'iter': OUT['iter_count'], 'times': TIMES, 'files': EXP['files'] if EXP else {},
            'opensim_ok': bool(EXP and EXP['ok']), 'figures': FIGS,
            'id_agreement': ({'ok': chk['id_tool']['agree'], 'min_corr': chk['id_tool']['min_corr'],
                              'max_rms_rel': chk['id_tool']['max_rms_rel']} if EXP else None),
            'filters': '追う入力の関節角度・速度・加速度に 6 Hz のフィルタ（加速度は速度のスプラインから）．opencap-processing の歩行の例と同じ．周期の制約はなし',
            'checks': ({k: RES[k] for k in ('peak_r', 'peak_l', 'mean_bw', 'dev_max', 'dev_joint')} if RES else
                       {'peak_r': nan, 'peak_l': nan, 'mean_bw': nan, 'dev_max': nan, 'dev_joint': '―'}),
            'title': '# %s の%sを，筋駆動の OpenSimAD で解いた結果' % (TRIAL, WHAT),
            'data_line': '- データ：%s の %s（OpenCap）．%s' % (os.path.basename(SESSION.rstrip('/')), TRIAL, FLOOR),
            'window_line': '- 区間：%s %.3f〜%.3f 秒（%.3f 秒．%s%s）' % (WHAT, T0, T1, T1 - T0, RULE,
                                                                    '．踵接地は Zeni らの方法' if SIDE else ''),
            'data_notes': ['- 6 節の確認：' + ('通った' if OK else '当たった（' + '／'.join(REASONS) + '）') +
                           '．確認の目安は暫定値であり，結果の妥当性を保証するものではない．',
                           '- 同じ設定でも，実行する環境によって IPOPT の道すじが変わり，結果が少し変わる．'],
            'environment': os.environ.get('OSAD_ENV_NOTE', 'Docker（使える CPU %d）' % len(os.sched_getaffinity(0)))}
with open(os.path.join(RESDIR, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(results_readme(INFO_RES) if EXP else
            '# %s の%s（%.3f〜%.3f 秒）\n\n%s．OpenSimAD が書いた結果のファイル（.mot，.npy）だけを入れました．求解：%s，%s 回\n'
            % (TRIAL, WHAT, T0, T1, WHY, OUT['return_status'], OUT['iter_count']))
shutil.copy(os.path.join(ROOT, 'runner.log'), os.path.join(RESDIR, 'runner.log'))
print('書き出したフォルダ（コンテナの中の作業のフォルダ）：%s' % RESDIR)

## 8. 保存する

書き出したフォルダ（README と IPOPT のログを含む）を `OUT_DIR` に写し，同じ中身の zip も作ります．
コンテナの中の作業のフォルダ（`/opt/osad`）は，コンテナを止めると消えます．

In [ ]:
# 8. 保存する
os.makedirs(OUT_DIR, exist_ok=True)
DST = os.path.join(OUT_DIR, NAME)
shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(RESDIR, DST)
ZIP = pack_results(RESDIR, os.path.join(OUT_DIR, NAME + '_results.zip'), top=NAME)
print('結果のフォルダ：%s' % DST)
print('結果の zip：%s（%.1f MB）' % (ZIP, os.path.getsize(ZIP) / 1e6))
display(pd.DataFrame({'秒': TIMES, '分': {k: v / 60 for k, v in TIMES.items()}}).round(1))

## 参考文献

- Uhlrich SD, Falisse A, Kidziński Ł, Muccini J, Ko M, Chaudhari AS, Hicks JL, Delp SL (2023) OpenCap: Human movement dynamics from smartphone videos. *PLOS Comput Biol* 19(10):e1011462. doi:10.1371/journal.pcbi.1011462
- Falisse A, Serrancolí G, Dembia CL, Gillis J, De Groote F (2019) Algorithmic differentiation improves the computational efficiency of OpenSim-based trajectory optimization of human movement. *PLOS ONE* 14(10):e0217730. doi:10.1371/journal.pone.0217730
- Zeni JA, Richards JG, Higginson JS (2008) Two simple methods for determining gait events during treadmill and overground walking using kinematic data. *Gait Posture* 27(4):710–714. doi:10.1016/j.gaitpost.2007.07.007